# Build Your Own Knowledge Graph!

## Goals :


1.   Create your own Knowledge graph
2.   Visualize and extract meaning from the graph

## What is a Knowledge Graph? (Review)

A **Knowledge Graph** is a structured way of representing information using a graph-based data model. It connects data points through relationships, providing context and enabling advanced querying and reasoning capabilities.

Key components of a knowledge graph include:

*   **Nodes (Entities):** These represent real-world objects, concepts, or abstract ideas. Examples could be people, organizations, locations, events, or programming languages. Each node typically has properties or attributes that describe it.
*   **Edges (Relationships/Predicates):** These describe the connections or interactions between two nodes. An edge always has a direction and a specific type of relationship. For example, 'Person A *works_at* Company X', where 'works_at' is the relationship.
*   **Relations (Labels for Edges):** These define the type of connection an edge represents. Relations clarify the meaning of the link between two entities.

**Usefulness for Reasoning:**
Knowledge graphs are powerful for reasoning because they explicitly model the relationships between entities. This structure allows for:
*   **Discovering hidden connections:** By traversing multiple edges, one can find indirect relationships that might not be obvious in a tabular dataset.
*   **Answering complex queries:** Instead of just looking up facts, knowledge graphs can answer questions about how things are connected, enabling more sophisticated analysis.
*   **Contextual understanding:** They provide rich context by linking disparate pieces of information, helping to make sense of complex data landscapes.

## Install Required Libraries


The following libraries are essential for building and querying our knowledge graph:

*   `pandas`: Used for efficient data manipulation and analysis, particularly for handling structured data like DataFrames.
*   `networkx`: A powerful Python library for the creation, manipulation, and study of the structure, dynamics, and functions of complex networks.
*   `SPARQLWrapper`: A Python wrapper around a SPARQL service that can be used to query local or remote SPARQL endpoints, enabling data retrieval from sources like Wikidata.
*   `matplotlib`: A comprehensive library for creating static, animated, and interactive visualizations in Python, which will be used for plotting the knowledge graph.

In [ ]:
!pip install pandas networkx SPARQLWrapper plotly

print("Libraries installed successfully.")

## Querying Wikidata

### Introduction to Data Collection

The first crucial step in building our knowledge graph is **Data Collection**. For this project, we will leverage **Wikidata** as our primary data source. Wikidata is a free and open knowledge base that acts as a central storage for the structured data of its Wikimedia sister projects, including Wikipedia, Wikivoyage, Wiktionary, and others. It provides a vast repository of entities and their relationships, making it an excellent resource for constructing a knowledge graph.

### Role of Wikidata and SPARQL

*   **Wikidata:** As a collaborative, multilingual, secondary database, Wikidata collects, organizes, and stores structured information from various sources. This makes it a rich and diverse source for entities (e.g., people, organizations, concepts) and the relationships between them.

*   **SPARQL:** To interact with Wikidata's structured data, we use **SPARQL (SPARQL Protocol and RDF Query Language)**. SPARQL is a W3C standard query language for RDF (Resource Description Framework), which is the underlying data model for Wikidata. It allows us to formulate precise queries to retrieve specific entities, attributes, and relationships from the Wikidata endpoint.

### Utility Functions: `build_query` and `sparql_df`

To streamline the process of querying Wikidata, we will be utilizing two helper functions:

*   `build_query`: This function is designed to construct SPARQL queries dynamically based on specified parameters. It helps in generating well-formed queries that can fetch the desired information from Wikidata.

*   `sparql_df`: This function takes a SPARQL query as input, executes it against the Wikidata SPARQL endpoint, and then processes the results. Critically, it converts the raw SPARQL results into a user-friendly **pandas DataFrame**, making the data easily accessible and ready for further processing and transformation into our knowledge graph structure.

## Querying Wikidata - Implement Query Functions


In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

# define the endpoint_url
endpoint_url = "https://query.wikidata.org/sparql"

# no need to edit this
def build_query(search_term, limit=100):
    """
    Stable Wikidata entity lookup via exact English label match.
    SPARQLWrapper-safe. No MWAPI. No timeouts.
    """

    query = f"""
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX wdt: <http://www.wikidata.org/prop/direct#>
    PREFIX wikibase: <http://wikiba.se/ontology#>
    PREFIX bd: <http://www.bigdata.com/rdf#>

    SELECT ?entity ?entityLabel ?url ?country ?countryLabel WHERE {{

      ?entity rdfs:label "{search_term}"@en .

      OPTIONAL {{ ?entity wdt:P856 ?url . }}
      OPTIONAL {{ ?entity wdt:P17 ?country . }}

      SERVICE wikibase:label {{
        bd:serviceParam wikibase:language "en".
      }}
    }}
    LIMIT {limit}
    """
    return query


# no need to edit this either
def sparql_df(query):
    """
    Executes a SPARQL query and returns the results as a Pandas DataFrame.
    """
    sparql = SPARQLWrapper(endpoint_url)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)
    results = sparql.query().convert()

    processed_results = []
    for result in results["results"]["bindings"]:
        row = {}
        for var, value_dict in result.items():
            value = value_dict["value"]
            if var == "aliases" and value:
                row[var] = value.split(',')
            else:
                row[var] = value
        processed_results.append(row)

    df = pd.DataFrame(processed_results)

    if 'aliases' in df.columns:
        df['aliases'] = df['aliases'].apply(lambda x: [x] if isinstance(x, str) and ',' not in x and x else x)

    return df

ModuleNotFoundError: No module named 'SPARQLWrapper'

## Querying Wikidata - User Search Term



In [ ]:
search_term = 'MIT' # you can change this to any entity you want to explore
print(f" chosen search term: {search_term}")

## Querying Wikidata - Execute Query and Clean Data

Execute the SPARQL query, fetch data from Wikidata, perform initial data cleaning, and display the first few rows of the resulting DataFrame.


In [ ]:
import pandas as pd

# generate the SPARQL query string
sparql_query = build_query(search_term)

# execute the generated SPARQL query
wd_df = sparql_df(sparql_query)

# The sparql_df function returns full URIs, so we need to extract the Q-number
if not wd_df.empty and 'entity' in wd_df.columns:
    wd_df['entity'] = wd_df['entity'].apply(lambda x: x.split('/')[-1] if isinstance(x, str) and 'http://www.wikidata.org/entity/' in x else x)

# fallback for empty DataFrame
if wd_df.empty:
    print("Warning: The SPARQL query returned an empty DataFrame. It might have timed out or your search term is too specific.\n")


print(wd_df)

## Querying Wikidata - Prepare for Graph Construction


Transform the collected and cleaned Wikidata DataFrame (`wd_df`) into a list of relations and then into a new DataFrame (`df_relations`) with 'Source', 'Target', and 'Relationship' columns, suitable for graph construction.


In [ ]:
# TODO : Build the relationships table
# should look like the output below

## Entity Span Identification


Implement the `extract_entities(text)` function using NER (Named Entity Recognition) to identify and extract entity spans from the input text


In [ ]:
def extract_entities(text):
    """
    Identifies and extracts entity spans from the input text using NER.
    Returns an example of NER output and includes a TODO for detailed implementation.
    """
    print(f"Processing text for NER: '{text[:50]}...'\n")

    # TODO: integrate a real NER model (e.g., SpaCy, Hugging Face Transformers)


## Collect Attention Between Entity Spans



### Understanding Attention as a Signal for Relationships

In the context of Natural Language Processing (NLP), particularly with transformer models, **attention mechanisms** are crucial for understanding how different parts of an input sequence relate to each other. Attention allows a model to weigh the importance of different words (tokens) in a sentence when processing another word.

For our knowledge graph construction, attention can serve as a powerful signal for identifying relationships between entities:

*   **Higher Attention, Stronger Connection**: If tokens belonging to two different entities consistently exhibit high attention scores towards each other, it suggests a strong semantic connection or interaction between those entities. For example, in the sentence "*Guido van Rossum* created *Python*", the tokens comprising "Guido van Rossum" would likely pay significant attention to "Python" and vice versa, indicating a "creator_of" or "created_by" relationship.

*   **Contextual Relevance**: Attention weights inherently capture the contextual relevance between words. When a model processes a specific entity, the attention it places on other entities in the sentence can reveal the nature of their relationship within that context. This is particularly useful for identifying implicit relationships that might not be explicitly stated by a simple verb.

By analyzing these attention patterns, especially the attention scores between token spans identified as entities, we can infer potential relationships (edges) for our knowledge graph. The strength of the attention can even be used as a proxy for the 'weight' or 'confidence' of that relationship.

In [ ]:
def get_attention_edges(text, entities):
    """
    Conceptual function to extract attention-based relationships between entities.
    This function would typically load a transformer model, process the text,
    and analyze attention weights between entity spans.

    Args:
        text (str): The input text string.
        entities (list): A list of dictionaries, where each dictionary represents
                         an entity with 'text', 'start', 'end', and 'label' keys.

    Returns:
        list: A list of dictionaries, where each dictionary represents a potential
              relationship (edge) based on attention, e.g.,
              {'Source': 'Entity A', 'Target': 'Entity B', 'Relationship': 'ATTENDS', 'Attention_Score': 0.8}
              For now, it returns a placeholder.
    """
    print(f"Processing text for attention-based edges: '{text[:50]}'...")
    print(f"Entities identified: {[e['text'] for e in entities]}")

    # TODO: implement actual attention extraction using a transformer model.
    # 1. tokenize the text.
    # 2. pass tokens through a pre-trained transformer model (e.g., from Hugging Face Transformers).
    # 3. extract attention weights from the model's output layers.
    # 4. map attention weights back to original text spans/entities.
    # 5. analyze attention scores between entity tokens to infer relationships.


## Aggregate Attention



### The Necessity of Aggregating Attention Across Heads and Layers

In transformer models, attention mechanisms are often organized into multiple "heads" and stacked across several "layers". Understanding why we need to aggregate attention scores from these different components is crucial for extracting meaningful relationship signals for knowledge graph construction:

1.  **Diverse Relationship Capture (Multiple Heads):** Each attention head within a transformer layer is designed to learn different aspects of relationships or dependencies within the input sequence. For example, one head might focus on syntactic dependencies (e.g., subject-verb agreement), another on semantic relationships (e.g., 'created by'), and yet another on co-reference (e.g., linking pronouns to their antecedents). By having multiple heads, the model can simultaneously attend to different contextual signals. Aggregating across these heads (e.g., by averaging or summing their attention scores) allows us to combine these diverse perspectives into a more comprehensive and robust measure of how two entities are related.

2.  **Hierarchical Feature Extraction (Multiple Layers):** Transformer models stack multiple layers, with each layer building upon the representations learned by the previous one. Lower layers tend to capture more local and syntactic information, while higher layers can model more abstract, global, and semantic relationships. To get a holistic view of the relationship between two entities, it's beneficial to consider the attention patterns from across all relevant layers. Aggregating attention across layers allows us to combine these hierarchical signals, providing a richer understanding of long-range and complex dependencies.

3.  **Robustness and Noise Reduction:** Individual attention heads or layers might occasionally focus on spurious correlations or exhibit noisy patterns. By aggregating attention scores across multiple heads and layers, we can smooth out these anomalies, amplify consistent signals, and derive a more stable and reliable indicator of true relationships between entities.

4.  **Simplified Relationship Signal:** For downstream tasks like knowledge graph construction, we often need a single, aggregated score representing the strength or nature of a relationship between two entities. Aggregation provides a way to distill the complex, multi-faceted attention information into a more manageable and interpretable form, suitable for inferring edges and their weights in a graph.

In [ ]:
import torch

def aggregate_attention(attn_tensor, method='mean'):
    """
    Aggregates attention scores across different heads and/or layers.

    Args:
        attn_tensor (torch.Tensor or np.ndarray): A tensor representing attention weights.
                                                   Expected shape: [num_layers, num_heads, sequence_length, sequence_length].
        method (str): The aggregation method ('mean', 'max', 'sum'). Defaults to 'mean'.

    Returns:
        torch.Tensor or np.ndarray: An aggregated attention tensor. For now, returns a placeholder.
    """
    print(f"Aggregating attention with method: {method}")
    print(f"Input attention tensor shape: {attn_tensor.shape if hasattr(attn_tensor, 'shape') else 'N/A'}")

    # TODO: implement actual attention aggregation logic here.

### Building the NetworkX Graph

Construct a directed graph using the `networkx` library. The `df_relations` DataFrame, which contains our extracted relationships, will be used to populate this graph. Each 'Source' and 'Target' in the DataFrame will become a node, and the 'Relationship' will define the edge connecting them.

In [ ]:
import networkx as nx

# create a directed graph
G = nx.DiGraph()

# iterate through df_relations and add nodes and edges
for index, row in df_relations.iterrows():
    source = row['Source']
    target = row['Target']
    relationship = row['Relationship']

    # add nodes
    G.add_node(source)
    G.add_node(target)

    # add edge with relationship
    G.add_edge(source, target, relationship=relationship)

print(f"Knowledge Graph created with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

## Visualize Knowledge Graph

Plot the constructed NetworkX graph `G` using `plotly`, displaying nodes, edges, and relationship labels to highlight key entities and their connections.


In [ ]:
import plotly.graph_objects as go
import networkx as nx

pos = nx.spring_layout(G, k=0.8, iterations=50)

# edge coords and labels
edge_x = []
edge_y = []
for edge in G.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x.append(x0)
    edge_x.append(x1)
    edge_x.append(None)
    edge_y.append(y0)
    edge_y.append(y1)
    edge_y.append(None)

# node coords and labels
node_x = []
node_y = []
node_labels = []
for node in G.nodes():
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)
    node_labels.append(node)

# create edge trace
edge_trace = go.Scatter(
    x=edge_x, y=edge_y,
    line=dict(width=1, color='gray'),
    hoverinfo='none',
    mode='lines')

# create node trace
node_trace = go.Scatter(
    x=node_x, y=node_y,
    mode='markers+text',
    hoverinfo='text',
    text=node_labels,
    textposition='top center',
    marker=dict(
        showscale=False,
        size=15,
        color='skyblue',
        line_width=2))

# create the final figure
fig = go.Figure(data=[edge_trace, node_trace],
             layout=go.Layout(
                title='<br>Interactive Knowledge Graph Visualization (Plotly)',
                titlefont_size=16,
                showlegend=False,
                hovermode='closest',
                margin=dict(b=20,l=5,r=5,t=40),
                annotations=[ dict(
                    text="",
                    showarrow=False,
                    xref="paper", yref="paper",
                    x=0.005, y=-0.002 ) ],
                xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
                yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)))

# display
fig.show()

## Thresholding and Entity-Level Graph Construction


### Explanation of Top-K Pruning and Collapsing Token Links into Entity Edges

#### Top-K Pruning

**Top-K pruning** is a technique used to refine the set of potential relationships (edges) identified from attention scores. When analyzing attention weights between tokens, especially in a transformer model, we often end up with a dense matrix of scores, where every token might have some attention towards every other token. Not all these connections are equally significant for forming a knowledge graph.

In the context of attention scores, top-K pruning involves:
*   **Filtering by Strength**: For each entity pair, or even for each token within an entity looking at tokens within another entity, we would calculate an aggregated attention score. "Top-K" then means keeping only the 'K' strongest relationships (those with the highest attention scores) between entities or within a predefined context (e.g., within a sentence or document).
*   **Reducing Noise**: This process helps in focusing on the most salient connections and discarding weak, noisy, or irrelevant attention signals. It acts as a form of regularization, preventing the knowledge graph from becoming too cluttered with spurious edges.
*   **Managing Complexity**: By selecting only the most significant links, top-K pruning reduces the computational complexity of subsequent graph processing steps and improves the interpretability of the resulting graph.

#### Collapsing Token Links into Entity Edges

**Collapsing token links into entity edges** is the process of transforming granular attention scores between individual tokens into meaningful relationships between higher-level entities. Transformer models operate at the token level, meaning attention scores are computed between individual words or subwords. However, a knowledge graph requires relationships between **entities** (e.g., 'Python' and 'Guido van Rossum'), which are often composed of multiple tokens.

The process typically involves:
1.  **Token-to-Entity Mapping**: First, tokens are mapped back to their respective entities. For example, if "Guido van Rossum" is an entity, all tokens within that phrase ("Guido", "van", "Rossum") are associated with that entity.
2.  **Aggregating Attention Scores**: Once tokens are mapped to entities, the attention scores between individual tokens belonging to different entities need to be aggregated to represent an overall attention score between the entities themselves. This aggregation can involve summing, averaging, or taking the maximum of the attention scores between all token pairs belonging to the two entities.
3.  **Inferring Entity-Level Relationships**: The aggregated attention score between two entities then serves as a strong indicator of a potential relationship. This single score can be used to decide whether an edge should be created between the two entities in the knowledge graph and, optionally, to assign a confidence weight to that edge.

In [ ]:
def build_entity_graph(entities, scores, k=3):
    """
    Constructs a graph by selecting top-K attention-based edges between entities
    and assigning confidence scores.

    Args:
        entities (list): A list of dictionaries, where each dictionary represents
                         an entity with 'text', 'start', 'end', and 'label' keys.
        scores (list): A list of potential relationship scores (e.g., aggregated attention scores)
                       between entities or token spans.
        k (int): The number of top relationships to keep (for top-K pruning).

    Returns:
        list: A list of dictionaries, where each dictionary represents an edge
              with 'Source', 'Target', 'Relationship', and 'Confidence' keys.
              For now, it returns a placeholder.
    """
    print(f"Building entity graph with {len(entities)} entities and {len(scores)} potential scores. Keeping top {k} edges.")

    # TODO: Implement logic to process 'scores' to infer relationships between 'entities'.
    # 1. mapping raw scores (e.g., token-level attention) to entity-level scores.
    # 2. aggregating scores for each potential entity-entity pair.
    # 3. applying top-K pruning: selecting the 'k' strongest relationships.
    # 4. assigning a 'Confidence' score to each selected relationship (edge).

    return []



## Provenance and Traceability


Define a `provenance` data structure to store metadata such as document ID, entity spans, layers used, and aggregation method.


### Importance of Provenance and Traceability in Knowledge Graph Construction

**Provenance** refers to the origin, source, and history of data. In the context of knowledge graph construction, traceability means being able to track how each piece of information (nodes and edges) in the graph came into existence, including its source, the methods used to extract it, and any transformations it underwent.

Its importance is paramount for several reasons:

1.  **Trust and Reliability**: Knowing the provenance of data allows users to assess the credibility and reliability of the information presented in the knowledge graph. If a fact can be traced back to a reputable source, confidence in that fact increases.
2.  **Debugging and Error Detection**: When anomalies or errors are found in the graph, provenance data can help pinpoint the exact source of the error – whether it's a faulty data source, an incorrect extraction rule, or an erroneous transformation step. This significantly speeds up debugging.
3.  **Understanding Data Quality**: Provenance metadata can include information about the confidence or certainty associated with an extracted fact, the context from which it was derived, and the algorithms used. This helps in understanding the overall quality and potential limitations of the graph.
4.  **Replicability and Reproducibility**: With detailed provenance, the process of constructing the knowledge graph can be fully replicated, ensuring that the same graph (or similar) can be generated given the same input data and processes. This is crucial for scientific research and enterprise-level data management.
5.  **Compliance and Auditing**: In many regulated industries, it's essential to demonstrate where data came from and how it was processed. Provenance provides the necessary audit trail for compliance requirements.
6.  **Evolvability and Maintenance**: As sources change or new information becomes available, provenance helps in identifying which parts of the graph need to be updated or re-evaluated, making the graph more maintainable and adaptable over time.

In [ ]:
# TODO : populate the list
provenance_data = {
}

print(provenance_data)

## Knowledge Graph Querying

Implement basic knowledge graph querying functionalities, including simple neighbor lookup and simple entity search.


### Querying the Knowledge Graph

For this initial phase of knowledge graph construction, we will focus on implementing basic querying functionalities. This involves:

*   **Neighbor Lookup**: Finding directly connected nodes to a given entity.
*   **Simple Entity Search**: Locating entities within the graph based on a search term.

More advanced querying capabilities, such as **pathfinding** between entities and complex **reasoning** over the graph structure, are crucial aspects of a fully functional knowledge graph. However, for the scope of this subtask, these features will be left as `TODO` items for future development. This allows us to establish the fundamental graph structure and basic interaction before diving into more sophisticated analytical tools.

In [ ]:
import networkx as nx

def find_neighbors(graph, node):
    """
    Finds the direct neighbors of a given node in a NetworkX graph.

    Args:
        graph (nx.Graph or nx.DiGraph): The NetworkX graph.
        node (str): The node for which to find neighbors.

    Returns:
        list: A list of direct neighbors of the node.
              Returns an empty list if the node is not in the graph.
    """
    if node in graph:
        return list(graph.neighbors(node))
    else:
        print(f"Warning: Node '{node}' not found in the graph.")
        return []

In [ ]:
def search_entity(graph, entity_name):
    """
    Searches for entities in the graph whose names contain the given search term (case-insensitive).

    Args:
        graph (nx.Graph or nx.DiGraph): The NetworkX graph.
        entity_name (str): The search term for entities.

    Returns:
        list: A list of nodes (entity names) that match the search term.
    """
    matching_nodes = []
    search_term_lower = entity_name.lower()
    for node in graph.nodes:
        if search_term_lower in str(node).lower():
            matching_nodes.append(node)
    if not matching_nodes:
        print(f"No entity found containing '{entity_name}'.")
    return matching_nodes

## Paths, Neighbors, and Multi-Hop Reasoning

Provide a conceptual explanation for paths, neighbors, and multi-hop reasoning in a knowledge graph. Include function stubs for these functionalities, marked as TODOs.


### Paths, Neighbors, and Multi-Hop Reasoning

#### Paths
A **path** in a knowledge graph represents a sequence of connected entities and relationships that link a starting entity to an ending entity. It's essentially a route through the graph. Paths can reveal indirect connections and causal chains that might not be immediately obvious from direct relationships. For example, a path might show how 'Person A' is connected to 'Project X' through 'Company B' and 'Department C'. Analyzing paths helps in understanding the flow of information, influence, or events within the knowledge domain.

#### Neighbors
While "neighbors" typically refer to directly connected entities, in a broader sense, it can also encompass entities that are within a certain 'distance' (number of hops) from a given node. For instance, "2-hop neighbors" would be entities reachable by traversing exactly two relationships. Understanding these extended neighborhoods can reveal communities, clusters, or indirect associations that are crucial for comprehensive analysis. Common neighbors between two entities can also indicate shared contexts or latent relationships.

#### Multi-Hop Reasoning
**Multi-hop reasoning** involves traversing multiple relationships (hops) in a knowledge graph to answer complex questions or infer new facts. It goes beyond simple lookup queries and uses the interconnected structure of the graph to deduce conclusions. For example, to answer "What programming languages are used by people who work for companies in the Netherlands?", one would need to perform several hops: from 'Netherlands' to 'Company X', then to 'Person Y' who 'works_for' 'Company X', and finally to 'Programming Language Z' that 'Person Y' 'uses'. This type of reasoning is fundamental for advanced AI applications, such as question answering, recommendation systems, and hypothesis generation, as it leverages the full semantic richness of the graph.

In [ ]:
import networkx as nx

def find_paths(graph, source, target):
    """
    Finds all paths between a source and a target node in a NetworkX graph.

    Args:
        graph (nx.Graph or nx.DiGraph): The NetworkX graph.
        source (str): The starting node.
        target (str): The ending node.

    Returns:
        list: A list of lists, where each inner list represents a path.
              For now, returns a placeholder.
    """
    print(f"Finding paths from '{source}' to '{target}'...")
    # TODO: implement actual pathfinding logic
    return []

def find_common_neighbors(graph, node1, node2):
    """
    Finds common neighbors between two nodes in a NetworkX graph.

    Args:
        graph (nx.Graph or nx.DiGraph): The NetworkX graph.
        node1 (str): The first node.
        node2 (str): The second node.

    Returns:
        list: A list of common neighbors.
              For now, returns a placeholder.
    """
    print(f"Finding common neighbors between '{node1}' and '{node2}'...")
    # TODO: implement actual common neighbor finding logic
    return []

def multi_hop_reasoning(graph, start_node, max_hops):
    """
    Performs multi-hop reasoning starting from a node up to a maximum number of hops.

    Args:
        graph (nx.Graph or nx.DiGraph): The NetworkX graph.
        start_node (str): The node to start reasoning from.
        max_hops (int): The maximum number of hops to traverse.

    Returns:
        dict: A dictionary representing reachable nodes and their paths/distances.
              For now, returns a placeholder.
    """
    print(f"Performing multi-hop reasoning from '{start_node}' for up to {max_hops} hops...")
    # TODO:  implement actual multi-hop reasoning logic
    return {}



## Relation Extraction


### How Relations Emerge and Are Labeled in a Knowledge Graph

In a knowledge graph, **relations** (or edges) fundamentally describe the connections and interactions between **entities** (or nodes). The process of establishing these relations involves two main stages:

#### 1. The Existence of a Relationship

The existence of a relationship between two entities is typically inferred from various sources:

*   **Structured Data**: In cases like Wikidata, relationships are explicitly defined and collected (e.g., 'Python *was_created_by* Guido van Rossum').
*   **Text Analysis (Implicit)**: From unstructured text, relationships are often implicitly present. Techniques like:
    *   **Named Entity Recognition (NER)** helps identify the entities themselves.
    *   **Coreference Resolution** links mentions of the same entity throughout a text.
    *   **Attention Mechanisms (Transformers)**: As discussed, high attention scores between tokens belonging to different entities can signal a strong connection or interaction between those entities. This often indicates *that a relationship exists*, even if its precise nature isn't immediately clear.
    *   **Dependency Parsing/Syntactic Analysis**: The grammatical structure of sentences can reveal how entities are related (e.g., subject-verb-object patterns).

Essentially, the first step identifies a pair of entities that are somehow connected or interacting within a given context.

#### 2. The Labeling of a Relationship

Once a relationship's existence is established, the next crucial step is to **label** it with a semantically meaningful predicate. This labeling transforms a generic connection into a specific, interpretable interaction, making the knowledge graph actionable for reasoning.

*   **From Explicit Predicates**: If the data source explicitly provides a relationship type (like Wikidata's properties), labeling is straightforward.
*   **From Textual Patterns**: For relationships extracted from text, labeling is more complex. It involves analyzing the words, phrases, and syntactic structures that connect the entities. For example:
    *   If "Guido van Rossum" and "Python" are connected by the verb "created", the relation might be labeled `CREATED_BY`.
    *   If "Company X" and "City Y" are connected by "located in", the relation might be `LOCATED_IN`.
*   **Machine Learning/Semantic Role Labeling**: Advanced NLP models can be trained to predict the most appropriate predicate label given the two entities and the connecting context. This can leverage features like dependency paths, semantic roles, and even the attention patterns themselves.

The challenge lies in moving from a raw indication of relatedness (e.g., two entities frequently co-occur or attend to each other) to a precise and consistent semantic label (e.g., `WORKS_FOR`, `FOUNDED`, `LOCATED_IN`). This often requires a combination of linguistic rules, pattern matching, and sophisticated machine learning models to capture the nuances of human language.

In [ ]:
def label_relation(source_entity, target_entity, edge_features):
    """
    Assigns a semantic label to a relationship (edge) between two entities.

    Args:
        source_entity (str): The text of the source entity.
        target_entity (str): The text of the target entity.
        edge_features (dict): A dictionary of features associated with the edge,
                              e.g., aggregated attention scores, dependency paths, etc.

    Returns:
        str: A semantic label for the relationship. For now, returns a placeholder.
    """
    print(f"Attempting to label relation between '{source_entity}' and '{target_entity}' with features: {edge_features}")

    # TODO: Implement relation labeling logic based on attention patterns, dependency parses,
    # or external knowledge.
    # - rule-based extraction
    # - machine learning models
    # - leveraging entity types
    return "UNKNOWN_RELATION"



## Neurosymbolic Reasoning


### Neurosymbolic Reasoning: Combining Neural Signals with Symbolic Constraints

**Neurosymbolic reasoning** is an emerging paradigm in artificial intelligence that seeks to combine the strengths of two historically distinct approaches: **neural networks** (or connectionist models) and **symbolic AI**. This hybrid approach aims to achieve both the robust pattern recognition and learning capabilities of neural models, and the logical consistency, interpretability, and reasoning power of symbolic systems.

#### Interplay Between Neural Signals and Symbolic Constraints

1.  **Neural Signals for Perception and Pattern Recognition**: Neural networks excel at processing raw, noisy data (like text, images, or sensor readings) to identify patterns, extract features, and make probabilistic predictions. In a neurosymbolic system, neural components act as the "perception" layer, converting unstructured or semi-structured data into a more abstract, semantic representation. For instance, a neural network might:
    *   Identify entities and relationships from natural language text (as we are doing with NER and attention).
    *   Extract properties from images or audio.
    *   Learn embeddings that capture semantic similarities.
    
    These outputs, often in the form of continuous vectors or probabilities, can be thought of as **"neural signals"** that suggest potential facts or hypotheses about the world.

2.  **Symbolic Constraints for Logical Reasoning and Consistency**: Symbolic AI, on the other hand, operates on discrete symbols and predefined rules. It provides mechanisms for logical inference, knowledge representation, and adherence to specific domain constraints. In a neurosymbolic system, symbolic components act as the "reasoning" layer, which:
    *   Takes the neural signals (e.g., predicted entities and relationships, possibly with confidence scores) as input.
    *   Applies logical rules, ontologies, and domain-specific constraints to these signals.
    *   Checks for consistency, infers new facts, or corrects inconsistent neural predictions.
    *   Can perform multi-hop reasoning, planning, and explain its conclusions.

    **"Symbolic constraints"** can include logical axioms (e.g., "if A is a parent of B, then B cannot be a parent of A"), type hierarchies (e.g., "a programming language is a type of software"), or domain-specific rules (e.g., "a person can only be born once"). These constraints ensure that the knowledge graph remains logically sound and adheres to known truths.

3.  **The Bidirectional Flow**: The power of neurosymbolic reasoning lies in the **interplay** and often **bidirectional flow** between these two components:
    *   Neural signals inform the symbolic system by providing candidates for facts and relationships.
    *   Symbolic constraints guide the neural learning process (e.g., through differentiable logic) or refine its outputs, ensuring that the neural predictions align with logical rules.
    *   This iterative process allows the system to leverage the best of both worlds: the flexibility and learning from data of neural networks, and the precision, interpretability, and generalizability of symbolic reasoning.

In [ ]:
def perform_neurosymbolic_reasoning(graph, neural_signals, symbolic_constraints):
    """
    A placeholder function for performing neurosymbolic reasoning.
    It conceptually outlines the interplay between neural signals and symbolic constraints
    for knowledge graph construction and reasoning.

    Args:
        graph (nx.Graph or nx.DiGraph): The current knowledge graph.
        neural_signals (dict or list): Outputs from neural models (e.g., predicted entities, relations,
                                      confidence scores, attention weights).
        symbolic_constraints (dict or list): Logical rules, ontologies, or domain-specific axioms.

    Returns:
        dict: A placeholder for reasoning results (e.g., inferred facts, consistency checks, updated graph).
              For now, returns a descriptive dictionary.
    """
    print("\n--- Performing Neurosymbolic Reasoning (Conceptual Placeholder) ---")
    print(f"Received graph with {graph.number_of_nodes()} nodes and {graph.number_of_edges()} edges.")
    print(f"Processing neural signals: {len(neural_signals)} items.")
    print(f"Applying symbolic constraints: {len(symbolic_constraints)} items.")

    # TODO: implement a full neurosymbolic reasoning engine here.
    # converting neural outputs (e.g., confidence-weighted entity/relation extractions) into symbolic facts or propositions.
    # integrating these facts with existing knowledge in the 'graph' under 'symbolic_constraints'.
    # applying logical rules and constraints to infer new facts, check for consistency, or resolve contradictions.
    # potentially feeding back symbolic results to refine neural models or explain decisions.

    return {
    }



## Conceptual Overview: IterE-Inspired Neuro-Symbolic KG Expansion

### Neuro-Symbolic Knowledge Graph Expansion
Knowledge graph expansion aims to enrich an existing knowledge graph (KG) by discovering new entities and relationships. Neuro-symbolic approaches combine the strengths of symbolic reasoning (logic, rules) with neural networks (pattern recognition, embeddings) to achieve more robust and accurate expansion. This fusion allows us to leverage both explicit domain knowledge and implicit patterns learned from data, leading to a powerful synergy for extending KGs.

### The IterE Workflow
The IterE approach, which inspires this conceptual overview, offers an elegant framework for iterative neuro-symbolic KG expansion. It effectively marries rule-based deduction with neural-network-based confidence scoring. Here's a high-level look at the workflow:

a.  **Initial Knowledge Graph**: The process begins with an existing knowledge graph, which might be incomplete but provides a foundational set of entities and relationships. This serves as the 'ground truth' for expansion.

b.  **Symbolic Rule Application**: Based on the current state of the knowledge graph, symbolic rules (often in the form of Horn clauses, e.g., "If A *is_friend_with* B AND B *is_friend_with* C, THEN A *is_friend_with* C") are applied. These rules act as logical templates to generate candidate triples (new potential relationships). For instance, if the KG contains (Guido van Rossum, *CREATED*, Python) and (Python, *HAS_TYPE*, Programming Language), a rule might suggest (Guido van Rossum, *CREATED*, Programming Language) as a candidate.

c.  **Neural Scoring**: Instead of directly adding these candidate triples to the KG, they are first evaluated by a neural model. This model, often trained on existing KG facts, learns to represent entities and relationships as numerical embeddings (vectors). It then uses these embeddings to calculate a confidence score for each candidate triple, indicating how likely it is to be true based on learned patterns.

d.  **Filtering and Expansion**: Candidate triples with a confidence score above a certain threshold are considered valid and are added to the knowledge graph. This step ensures that only the most probable new facts enrich the graph, preventing the introduction of noisy or incorrect information.

e.  **Iteration**: The expanded knowledge graph then becomes the new 'initial' graph for the next iteration. The symbolic rules are re-applied, new candidate triples are generated, neural models re-score them, and the graph is further expanded. This iterative loop allows for continuous growth and refinement of the knowledge graph.

By integrating symbolic rule-based generation with neural confidence scoring, neuro-symbolic approaches like IterE combine the precision and interpretability of logical rules with the generalization and robustness of machine learning. This powerful combination helps overcome the limitations of purely symbolic or purely neural systems, enabling more effective and accurate knowledge graph expansion.

## Knowledge Graph Representation and Initial Data

### Subtask:
Explain how the knowledge graph will be represented, define a small initial knowledge graph, and load this data into a suitable Python structure.


### Knowledge Graph Representation

Our knowledge graph will be represented as a collection of **triples**, which are fundamental units composed of three parts: a **head** entity, a **relation** (or predicate), and a **tail** entity. This structure effectively describes a relationship between two entities.

Each triple can be conceptualized as `(Subject, Predicate, Object)` or `(Head, Relation, Tail)`. For example:

*   `('Guido van Rossum', 'CREATED', 'Python')`
    *   **Head (Subject):** Guido van Rossum
    *   **Relation (Predicate):** CREATED
    *   **Tail (Object):** Python

This simple yet powerful structure allows us to build a network of interconnected information, where entities are nodes and relations are directed edges.

In [ ]:
import pandas as pd

# TODO: Define a small initial knowledge graph as a list of tuples

# TODO: Convert the initial_kg_triples list into a pandas DataFrame

print("First 5 rows of the initial knowledge graph DataFrame:")


## Symbolic Reasoning: Horn-Clause Rule Implementation


### Horn-Clause Rules and Symbolic Reasoning

**Horn-clause rules** are a fundamental concept in symbolic AI and logic programming, often used to represent logical implications. They are a specific type of logical clause (a disjunction of literals) where at most one of the literals is positive (unnegated).

In the context of knowledge graphs, Horn-clause rules allow us to infer new facts (triples) from existing ones. They typically take the form of `Head :- Body`, which can be read as "Head is true if Body is true." The `Body` consists of one or more positive literals (conjunction of facts), and the `Head` is a single positive literal. If all conditions in the `Body` are met by facts already present in the knowledge graph, then the `Head` (a new fact) can be logically deduced.

**Role in Symbolic Reasoning:**
Horn-clause rules are crucial for symbolic reasoning because they enable automatic deduction and knowledge expansion. They provide a structured way to define domain-specific axioms and common-sense knowledge. By applying these rules, a knowledge graph can grow by inferring relationships that are logically implied by its current state, rather than solely relying on explicit extraction from text.

**Example Format for Hard-Coded Rules:**
For our purposes, we can represent Horn-clause rules as tuples or dictionaries that define how new relations can be inferred. A common type of rule is transitivity or composition.

Consider a rule like:
`IF (A, R1, B) AND (B, R2, C) THEN (A, R3, C)`

This rule states: If entity `A` has a relation `R1` with `B`, and `B` has a relation `R2` with `C`, then we can infer that `A` has a relation `R3` with `C`.

We can represent such a rule programmatically as a tuple:
`('R1', 'R2', 'R3')`

Or, more explicitly, for readability:
`{
    'antecedent1_relation': 'R1',
    'antecedent2_relation': 'R2',
    'consequent_relation': 'R3'
}`

**Examples of such rules:**
*   **Transitivity:** `(IS_LOCATED_IN, IS_PART_OF, IS_LOCATED_IN)`
    *   If (City, IS_LOCATED_IN, State) AND (State, IS_PART_OF, Country) THEN (City, IS_LOCATED_IN, Country)
*   **Symmetry (a special case can be handled by rules):** If (A, HAS_CHILD, B) THEN (B, HAS_PARENT, A)
*   **Inverse Relationships:** `(CREATED_BY, HAS_CREATOR, CREATED_BY_INVERSE)` (Where CREATED_BY_INVERSE would be a new relation 'HAS_CREATED')
    *   If (Person, CREATED_BY, Software) THEN (Software, HAS_CREATOR, Person)

These rules will be applied to our `kg_df` to discover new potential facts.

In [ ]:
import pandas as pd

def apply_rules(kg_df, rules):
    """
    Applies Horn-clause rules to the knowledge graph DataFrame to generate new candidate triples.

    Args:
        kg_df (pd.DataFrame): The current knowledge graph with 'head', 'relation', 'tail' columns.
        rules (list): A list of Horn-clause rules. Each rule is a dictionary defining
                      the type and predicates involved.

    Returns:
        set: A set of new candidate triples (head, relation, tail) not already in kg_df.
    """
    candidate_triples = set()
    existing_triples = set(tuple(x) for x in kg_df.to_numpy())

    print(f"Starting with {len(existing_triples)} existing triples.")

    for rule in rules:
        rule_type = rule['type']

        if rule_type == 'transitive':
            R1, R2, R3 = rule['R1'], rule['R2'], rule['R3']
            df_R1 = kg_df[kg_df['relation'] == R1]
            df_R2 = kg_df[kg_df['relation'] == R2]

            merged_df = pd.merge(df_R1, df_R2, left_on='tail', right_on='head', suffixes=('_R1', '_R2'))

            for _, row in merged_df.iterrows():
                new_head = row['head_R1']
                new_tail = row['tail_R2']
                new_triple = (new_head, R3, new_tail)
                if new_triple not in existing_triples:
                    candidate_triples.add(new_triple)

        elif rule_type == 'inverse':
            R1, R2 = rule['R1'], rule['R2']
            df_R1 = kg_df[kg_df['relation'] == R1]

            for _, row in df_R1.iterrows():
                new_head = row['tail']
                new_tail = row['head']
                new_triple = (new_head, R2, new_tail)
                if new_triple not in existing_triples:
                    candidate_triples.add(new_triple)

        elif rule_type == 'alias_inference':
            R1, R2, R3 = rule['R1'], rule['R2'], rule['R3']
            df_R1 = kg_df[kg_df['relation'] == R1]
            df_R2 = kg_df[kg_df['relation'] == R2]

            merged_df = pd.merge(df_R1, df_R2, left_on='tail', right_on='head', suffixes=('_alias', '_original'))

            for _, row in merged_df.iterrows():
                new_head = row['head_alias']
                new_tail = row['tail_original']
                new_triple = (new_head, R3, new_tail)
                if new_triple not in existing_triples:
                    candidate_triples.add(new_triple)

    return candidate_triples

# TODO: define example Horn-clause rules

# TODO: apply the rules to the kg_df to generate candidate triples


## Neural Component: Embedding-Based Edge Scoring

### Neural Embeddings: Representing Meaning in Vector Space

**Neural embeddings** are dense, low-dimensional vector representations of entities (like people, organizations, concepts) and relations (like 'CREATED_BY', 'LOCATED_IN') in a continuous vector space. The core idea is that semantically similar entities or relations are mapped to points that are close to each other in this vector space.

**Why use embeddings in Knowledge Graphs?**

1.  **Semantic Representation**: Embeddings capture the latent semantic features of entities and relations, allowing models to understand their meanings and relationships beyond simple string matching.
2.  **Scalability**: Instead of dealing with sparse, high-dimensional representations (like one-hot encodings), embeddings offer a compact way to represent knowledge, making computations more efficient.
3.  **Generalization**: Models trained on embeddings can generalize to unseen entities or relations if they share semantic properties with known ones, leading to better link prediction and reasoning.
4.  **Neural Scoring**: Embeddings are a natural fit for neural networks, enabling the use of mathematical operations (like dot products, distances) to quantify the likelihood of a relationship existing between two entities.

**How are Embeddings Typically Generated?**

Often, embeddings are learned through neural network training on large datasets (e.g., text corpora, existing knowledge graphs) where the model tries to predict missing links or classify relationships. Popular models include Word2Vec for words, and TransE, DistMult, ComplEx for knowledge graph entities and relations.

**For this Assignment: Lightweight Embeddings**

For the purpose of this assignment, to keep the focus on the neuro-symbolic workflow rather than complex model training, we will use **lightweight embeddings**.

*   **Randomly Initialized Vectors**: If an entity or relation does not yet have an embedding, we will simply initialize it with a random vector (e.g., using `numpy.random.rand`). This serves as a conceptual placeholder for a learned embedding.
*   **Fixed Dimension**: All embeddings (for both entities and relations) will have a fixed dimension (e.g., 50).

This approach allows us to simulate the presence of embeddings and demonstrate the scoring mechanism without the computational overhead of training a full-fledged embedding model.

In [ ]:
import numpy as np

# dictionary to store embeddings
embeddings_dict = {}

def get_or_create_embedding(entity_or_relation_name, embedding_dim=50):
    """
    Returns an embedding vector for a given entity or relation name.
    If the entity/relation already has an embedding, it returns it;
    otherwise, it creates a new random embedding and stores it.
    """
    global embeddings_dict

    if entity_or_relation_name not in embeddings_dict:
        embeddings_dict[entity_or_relation_name] = np.random.rand(embedding_dim)
        # normalize the embedding to have unit length, common practice in KGE
        embeddings_dict[entity_or_relation_name] = embeddings_dict[entity_or_relation_name] / np.linalg.norm(embeddings_dict[entity_or_relation_name])
    return embeddings_dict[entity_or_relation_name]

In [ ]:
def score_edge(head, relation, tail, embeddings_dict, embedding_dim=50):
    """
    Scores a candidate edge (head, relation, tail) using their embeddings.
    A simple TransE-inspired scoring mechanism: ||head_embedding + relation_embedding - tail_embedding||^2
    Lower score indicates higher confidence.
    """
    h_embed = get_or_create_embedding(head, embedding_dim)
    r_embed = get_or_create_embedding(relation, embedding_dim)
    t_embed = get_or_create_embedding(tail, embedding_dim)

    score = np.linalg.norm(h_embed + r_embed - t_embed)

    confidence = 1 / (1 + score)

    return confidence

In [ ]:
unique_entities_relations = set()

# TODO: add entities and relations from kg_df
# TODO: add entities and relations from candidate_triples
# TODO: populate embeddings_dict

print(f"Embeddings created for {len(embeddings_dict)} unique entities and relations.")


In [ ]:
print("\n--- Demonstrating score_edge function ---")

example_triples = list(candidate_triples)[:2]

if not example_triples:
    print("No candidate triples to demonstrate scoring.")
else:
    for i, (head, relation, tail) in enumerate(example_triples):
        confidence_score = score_edge(head, relation, tail, embeddings_dict)
        print(f"\nExample Candidate Triple {i+1}: ({head}, {relation}, {tail})")
        print(f"  Confidence Score: {confidence_score:.4f}")

        print(f"  Embedding for Head ('{head}'): {embeddings_dict[head][:5]}...")
        print(f"  Embedding for Relation ('{relation}'): {embeddings_dict[relation][:5]}...")
        if tail is not None:
            print(f"  Embedding for Tail ('{tail}'): {embeddings_dict[tail][:5]}...")
        else:
            print("  Tail is None, no embedding to show.")


## Iterative Expansion Loop

### Subtask:
Implement the iterative expansion loop, which involves applying symbolic rules to generate candidate triples, scoring these candidates using neural embeddings, and adding high-confidence triples to the knowledge graph. The loop should handle a confidence threshold and termination conditions.


### The Iterative Expansion Loop

The knowledge graph expansion process is designed to be iterative, meaning it continuously refines and grows the graph over several cycles. Each iteration builds upon the knowledge acquired in the previous one, aiming to discover new, high-confidence facts.

Here's how the iterative loop works:

1.  **Symbolic Rule Application (Candidate Generation)**: In each iteration, we start by applying a set of predefined **Horn-clause rules** to the *current* state of the knowledge graph. These rules act as logical templates, suggesting potential new relationships based on existing facts. For example, if we know A `IS_LOCATED_IN` B and B `IS_PART_OF` C, a rule might suggest A `IS_LOCATED_IN` C. The output of this step is a collection of **candidate triples**, which are potential new facts that might be added to the graph.

2.  **Neural Scoring (Confidence Assignment)**: Not all candidate triples generated by symbolic rules are equally reliable. To address this, each candidate triple is fed into a **neural component** that assigns a **confidence score** to it. This scoring typically involves using entity and relation embeddings (vector representations) to estimate the likelihood of the triple being true based on patterns learned from the existing graph. A higher score indicates a stronger belief in the truthfulness of the candidate.

3.  **Filtering by Confidence Threshold**: A crucial step is to filter these candidates. We use a predetermined **confidence threshold**. Only those candidate triples whose neural score exceeds this threshold are considered sufficiently reliable to be added to the knowledge graph. This prevents the graph from being polluted by low-confidence or incorrect inferences.

4.  **Knowledge Graph Expansion**: The high-confidence, filtered candidate triples are then added to the knowledge graph, expanding its size and interconnectedness. This expanded graph then becomes the basis for the next iteration.

5.  **Iteration and Termination Conditions**: The process repeats for a fixed number of iterations (`max_iterations`). The loop can also terminate early if, in a given iteration, no new valid triples are found. This indicates that the symbolic rules and current neural model are no longer able to generate sufficiently confident new facts, or that the graph has reached a stable state with respect to the given rules and threshold.

This iterative neuro-symbolic approach allows for a controlled and robust expansion of the knowledge graph, leveraging both logical deduction and learned semantic patterns.

In [ ]:
import pandas as pd
import numpy as np

confidence_threshold = 0.7
max_iterations = 5

print(f"Starting iterative KG expansion with threshold={confidence_threshold} and max_iterations={max_iterations}\n")

existing_triples_set = set(tuple(x) for x in kg_df[['head', 'relation', 'tail']].to_numpy())

for i in range(max_iterations):
    print(f"--- Iteration {i+1}/{max_iterations} ---")

    candidate_triples = apply_rules(kg_df, horn_clause_rules) # <-- you need to define

    if not candidate_triples:
        print("No new candidate triples generated. Terminating expansion.")
        break

    print(f"Generated {len(candidate_triples)} candidate triples.")

    new_valid_triples = []
    scored_candidates = []

    for head, relation, tail in candidate_triples:
        get_or_create_embedding(head)
        get_or_create_embedding(relation)
        if tail is not None:
            get_or_create_embedding(tail)

        confidence_score = score_edge(head, relation, tail, embeddings_dict)
        scored_candidates.append((head, relation, tail, confidence_score))

        if confidence_score > confidence_threshold:
            new_triple = (head, relation, tail)
            if new_triple not in existing_triples_set:
                new_valid_triples.append(new_triple)
                existing_triples_set.add(new_triple)

    if new_valid_triples:
        new_triples_df = pd.DataFrame(new_valid_triples, columns=['head', 'relation', 'tail'])
        kg_df = pd.concat([kg_df, new_triples_df], ignore_index=True)

        for head, relation, tail in new_valid_triples:
            get_or_create_embedding(head)
            get_or_create_embedding(relation)
            if tail is not None:
                get_or_create_embedding(tail)

        print(f"Added {len(new_valid_triples)} new valid triples.")
    else:
        print("No new valid triples met the confidence threshold. Terminating expansion.")
        break

    print(f"Total triples in KG after iteration {i+1}: {len(kg_df)}\n")

print("KG Expansion process completed.")
print(f"Final KG size: {len(kg_df)} triples.")
print(f"Final embeddings count: {len(embeddings_dict)} unique entities/relations.")

## Downstream Task: Edge Prediction / KG Completion Evaluation

### Explanation of Edge Prediction
**Edge Prediction**, also known as **Knowledge Graph Completion**, is a crucial downstream task for evaluating the quality and effectiveness of a knowledge graph expansion system. The goal of edge prediction is to predict missing links (relationships) between entities in a knowledge graph.

In the context of our neuro-symbolic expansion, this means assessing how well our system can infer new, previously unknown, but valid relationships. If our expansion process is effective, it should generate high-confidence candidate triples that correctly represent missing facts.

**How it works for evaluation:**
1.  **Hold-out Testing**: A common approach is to split an existing knowledge graph into training and testing sets. The training set is used to build or expand the KG, while the test set contains triples that were intentionally withheld. These withheld triples represent the 'missing edges' we want to predict.
2.  **Candidate Generation and Scoring**: During evaluation, the system attempts to predict these missing edges. This often involves generating candidate triples (e.g., using symbolic rules or neural models) and then scoring them based on their likelihood of being true.
3.  **Ranking and Metrics**: The true missing triples are then ranked against a set of *negative samples* (false triples). Evaluation metrics like **Hits@K**, Mean Rank (MR), or Mean Reciprocal Rank (MRR) are used to measure how highly the true triples are ranked compared to false ones. A higher Hits@K, for instance, means the true triple was found among the top K predictions more often.

This task is vital because it directly measures the system's ability to generalize and discover new, accurate information, which is a primary objective of knowledge graph expansion.

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd

# TODO: convert initial_kg_triples to a list of lists, handling None values gracefully
# Also, ensure all elements are strings for consistency in the split
# TODO: initial_kg_triples into training and test sets (e.g., 80/20 split)
# TODO: convert back to list of tuples for test_triples to match original format (can contain None)
# TODO: create kg_df_train DataFrame from train_triples

In [ ]:
import random
import numpy as np

def evaluate_expansion(expanded_kg_df, test_triples, k=5, num_negative_samples=10):
    """
    Evaluate an expanded knowledge graph using Hits@K.
    """

    # TODO: initialize counters and bookkeeping for Hits@K evaluation

    # TODO: extract the set of all entities from the expanded KG
    # (hint: entities appear in both head and tail columns)

    # TODO: handle edge cases where no entities or no test triples exist

    # TODO: build a fast lookup structure for existing triples
    # (used to avoid sampling false negatives)

    # TODO: iterate over each true test triple
    for true_triple in test_triples:
        # TODO: unpack (head, relation, tail)

        # TODO: ensure embeddings exist for entities and relations
        # (do NOT implement embedding logic here)

        # TODO: initialize candidate triples list with the true triple

        # TODO: generate negative samples by corrupting head or tail
        # - decide when to corrupt head vs tail
        # - avoid creating triples already in the KG

        # TODO: score all candidate triples using the embedding-based scoring function

        # TODO: rank candidate triples by score (higher = better)

        # TODO: determine the rank of the true triple

        # TODO: update Hits@K count if true triple is in top-K

    # TODO: compute final Hits@K metric

    # TODO: return Hits@K score
    pass


# -------------------------
# Run evaluation
# -------------------------

# TODO: ensure test triples are in (head, relation, tail) tuple format

print("\n--- Starting Knowledge Graph Evaluation ---")

# TODO: call evaluate_expansion with appropriate arguments

# TODO: print the final Hits@K score

IndentationError: expected an indented block after 'for' statement on line 20 (ipython-input-798319540.py, line 43)

# Part 3: Advanced Knowledge Graph Construction with Graphusion

## Motivation

Academic knowledge graphs typically contain papers, authors, and citations (heterogeneous graph), but lack detailed concept-level relationships. We'll use the **Graphusion algorithm** to extract a homogeneous concept graph from paper abstracts, then combine it with the heterogeneous structural graph.

## Graphusion Algorithm Overview

**Paper:** [GraphFusion: A Multi-Source Knowledge Graph Augmentation Method](https://arxiv.org/pdf/2407.10794)

### Three-Step Pipeline:

**Step 1: Seed Entity Generation**
- Use BERTopic for topic modeling on paper abstracts
- Extract seed entities (concepts/keywords) from each topic cluster
- These seeds serve as query concepts for triplet extraction

**Step 2: Candidate Triplet Extraction**
- For each seed entity, use it as a query
- Extract fine-grained triplets: (head_entity, relation, tail_entity)
- Use Chain-of-Thought prompting with LLMs
- Support 7 relation types:
  - Prerequisite_of
  - Used_for
  - Compare
  - Conjunction
  - Hyponym_of (is-a relationship)
  - Evaluate_for
  - Part_of

**Step 3: Knowledge Graph Fusion**
- **Entity Merging:** Merge semantically similar entities (e.g., "NMT" and "neural machine translation")
- **Conflict Resolution:** For each entity pair, keep only one relation (the most accurate)
- **Novel Triplet Inference:** Discover new relationships by examining uncovered entity pairs

### Implementation Plan

1. Load HTAG dataset (Cloudy1225/HTAG) - only document content (id, title, abstract)
2. Build homogeneous concept graph using Graphusion (Steps 1-3)
3. Load structural data (authors, citations, FoS) from arxiv.pkl
4. Combine concept graph with heterogeneous graph



## Exercise 0: Understanding Graphusion

Read the GraphFusion paper: [GraphFusion: A Multi-Source Knowledge Graph Augmentation Method](https://arxiv.org/pdf/2407.10794)

**Task:**
1. What are the three main steps of the Graphusion algorithm?
2. What are the 7 relation types used in the paper?
3. Why is BERTopic used for seed entity generation?


**Summary:** [Your Answer Here]

In [ ]:
# Install dependencies
!pip install datasets huggingface_hub openai anthropic aiohttp aiolimiter python-dotenv

import sys
import os
import asyncio
import json
import networkx as nx
import logging
# Basic logging setup
logging.basicConfig(level=logging.INFO)


In [ ]:
# [SETUP] Create a .env file for API keys
# Please fill in your API key below and run this cell.
env_content = '''
OPENAI_API_KEY=sk-...
OPENAI_MODEL=gpt-4o-mini
USE_PROVIDER=openai
# ANTHROPIC_API_KEY=sk-ant-...
'''

with open('.env', 'w') as f:
    f.write(env_content)

# Load environment
from dotenv import load_dotenv
load_dotenv(override=True)
print(".env file created and loaded.")


In [ ]:
"""
LLM Client for batch inference
"""

import os
import asyncio
from typing import List, Dict, Any, Optional
from dataclasses import dataclass
import logging
from openai import AsyncOpenAI
import anthropic
from aiolimiter import AsyncLimiter

logger = logging.getLogger(__name__)

@dataclass
class LLMConfig:
    """Configuration for LLM inference"""
    provider: str = "openai"
    model: str = "gpt-4o-mini"
    api_key: Optional[str] = None
    base_url: Optional[str] = None
    batch_size: int = 10
    max_concurrent: int = 5
    timeout: int = 60
    max_requests_per_minute: int = 500
    max_tokens_per_minute: int = 90000


class LLMClient:
    """Unified LLM client for batch inference"""

    def __init__(self, config: Optional[LLMConfig] = None):
        if config is None:
            config = self._load_config_from_env()

        self.config = config
        self.client = None

        # Rate limiters
        self._request_limiter = AsyncLimiter(
            max_rate=self.config.max_requests_per_minute,
            time_period=60
        )
        self._token_limiter = AsyncLimiter(
            max_rate=self.config.max_tokens_per_minute,
            time_period=60
        )

        self._setup_client()

    def _load_config_from_env(self) -> LLMConfig:
        """Load configuration from environment variables"""
        # Default to OpenAI for this tutorial
        return LLMConfig(
            provider="openai",
            model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"),
            api_key=os.getenv("OPENAI_API_KEY"),
            # Allow base_url override for compatible endpoints
            base_url=os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1"),
            batch_size=int(os.getenv("BATCH_SIZE", "10")),
            max_concurrent=int(os.getenv("MAX_CONCURRENT_REQUESTS", "5"))
        )

    def _setup_client(self):
        """Initialize the appropriate client"""
        if self.config.provider == "openai":
            self.client = AsyncOpenAI(
                api_key=self.config.api_key,
                base_url=self.config.base_url,
                timeout=self.config.timeout,
                max_retries=3
            )
            logger.info(f"Initialized OpenAI client")
        elif self.config.provider == "anthropic":
            self.client = anthropic.AsyncAnthropic(
                api_key=self.config.api_key,
                timeout=self.config.timeout,
                max_retries=3
            )
            logger.info(f"Initialized Anthropic client")

    # Keep the rest of the methods as they are generic
    async def _call_openai(self, prompt: str, system_prompt: str = "") -> str:
        """Call OpenAI API with rate limiting (following External Provider pattern)"""
        messages = []
        if system_prompt:
            messages.append({"role": "system", "content": system_prompt})
        messages.append({"role": "user", "content": prompt})

        max_retries = 3
        retry_count = 0

        async with self._request_limiter:
            while True:
                try:
                    # Build request parameters
                    request_params = {
                        "model": self.config.model,
                        "messages": messages
                    }

                    # Don't set temperature - many newer models don't support custom values
                    # They will use their default temperature

                    response = await self.client.chat.completions.create(**request_params)
                    return response.choices[0].message.content
                except Exception as e:
                    # Handle rate limits and server errors with retry
                    error_str = str(e)
                    if ("429" in error_str or "rate" in error_str.lower() or
                        "500" in error_str or "503" in error_str):
                        retry_count += 1
                        if retry_count >= max_retries:
                            logger.error(f"Max retries exceeded: {e}")
                            raise

                        delay = 2 ** retry_count
                        logger.warning(f"Rate limit/server error, retrying in {delay}s: {e}")
                        await asyncio.sleep(delay)
                    else:
                        logger.error(f"Error calling OpenAI API: {e}")
                        raise

    async def _call_anthropic(self, prompt: str, system_prompt: str = "") -> str:
        """Call Anthropic API with rate limiting"""
        max_retries = 3
        retry_count = 0

        async with self._request_limiter:
            while True:
                try:
                    message = await self.client.messages.create(
                        model=self.config.model,
                        max_tokens=4096,
                        system=system_prompt if system_prompt else None,
                        messages=[{"role": "user", "content": prompt}],
                        temperature=0.0
                    )
                    return message.content[0].text
                except Exception as e:
                    error_str = str(e)
                    if ("429" in error_str or "rate" in error_str.lower() or
                        "500" in error_str or "503" in error_str):
                        retry_count += 1
                        if retry_count >= max_retries:
                            logger.error(f"Max retries exceeded: {e}")
                            raise

                        delay = 2 ** retry_count
                        logger.warning(f"Rate limit/server error, retrying in {delay}s: {e}")
                        await asyncio.sleep(delay)
                    else:
                        logger.error(f"Error calling Anthropic API: {e}")
                        raise

    async def generate(self, prompt: str, system_prompt: str = "") -> str:
        """Generate a single response"""
        if self.config.provider == "openai":
            return await self._call_openai(prompt, system_prompt)
        elif self.config.provider == "anthropic":
            return await self._call_anthropic(prompt, system_prompt)
        else:
            raise ValueError(f"Unsupported provider: {self.config.provider}")

    async def generate_batch(
        self,
        prompts: List[str],
        system_prompt: str = "",
        show_progress: bool = True
    ) -> List[str]:
        """
        Generate responses for a batch of prompts with concurrency control
        Following batch processing pattern

        Args:
            prompts: List of prompts to process
            system_prompt: System prompt to use for all requests
            show_progress: Whether to show progress bar

        Returns:
            List of generated responses in the same order as input prompts
        """
        semaphore = asyncio.Semaphore(self.config.max_concurrent)

        async def generate_with_semaphore(prompt: str, index: int) -> tuple[int, str]:
            async with semaphore:
                try:
                    result = await self.generate(prompt, system_prompt)
                    return index, result
                except Exception as e:
                    logger.error(f"Error processing prompt {index}: {str(e)}")
                    return index, ""

        # Create tasks with their indices to maintain order
        tasks = [
            generate_with_semaphore(prompt, i)
            for i, prompt in enumerate(prompts)
        ]

        # Execute with progress tracking if requested
        if show_progress:
            try:
                from tqdm.asyncio import tqdm as async_tqdm
                results = await async_tqdm.gather(*tasks, desc="Processing prompts")
            except ImportError:
                logger.warning("tqdm not available, processing without progress bar")
                results = await asyncio.gather(*tasks)
        else:
            results = await asyncio.gather(*tasks)

        # Sort by index and extract responses
        results.sort(key=lambda x: x[0])
        return [response for _, response in results]

    async def extract_concepts_from_text(
        self,
        texts: List[str],
        max_concepts: int = 10
    ) -> List[Dict[str, Any]]:
        """
        Extract key concepts from a list of texts

        Args:
            texts: List of text documents
            max_concepts: Maximum number of concepts to extract per document

        Returns:
            List of dictionaries containing extracted concepts
        """
        system_prompt = """You are an expert at extracting key concepts from academic papers.
Extract the most important concepts, methods, and terms from the given text.
Return ONLY a JSON object with this structure:
{
  "concepts": [
    {"name": "concept name", "type": "method|algorithm|dataset|metric|task", "confidence": 0.0-1.0}
  ]
}"""

        prompts = [
            f"Extract up to {max_concepts} key concepts from this text:\n\n{text[:2000]}"
            for text in texts
        ]

        responses = await self.generate_batch(prompts, system_prompt, show_progress=True)

        # Parse JSON responses
        results = []
        import json
        for i, response in enumerate(responses):
            try:
                # Try to extract JSON from response
                if "```json" in response:
                    json_str = response.split("```json")[1].split("```")[0].strip()
                elif "```" in response:
                    json_str = response.split("```")[1].split("```")[0].strip()
                else:
                    json_str = response.strip()

                data = json.loads(json_str)
                results.append({
                    "text_index": i,
                    "concepts": data.get("concepts", [])
                })
            except Exception as e:
                logger.error(f"Error parsing response {i}: {str(e)}")
                results.append({"text_index": i, "concepts": []})

        return results

    async def extract_concept_relations(
        self,
        concepts: List[str]
    ) -> List[Dict[str, Any]]:
        """
        Extract relationships between concepts

        Args:
            concepts: List of concept names

        Returns:
            List of relationships as dictionaries
        """
        if len(concepts) < 2:
            return []

        system_prompt = """You are an expert at identifying relationships between concepts.
Given a list of concepts, identify meaningful relationships between them.
Return ONLY a JSON object with this structure:
{
  "relations": [
    {"source": "concept1", "target": "concept2", "relation": "relation_type", "confidence": 0.0-1.0}
  ]
}
Relation types: uses, improves, requires, evaluates_on, similar_to, extends"""

        prompt = f"""Identify relationships between these concepts:\n{', '.join(concepts)}"""

        response = await self.generate(prompt, system_prompt)

        try:
            import json
            if "```json" in response:
                json_str = response.split("```json")[1].split("```")[0].strip()
            elif "```" in response:
                json_str = response.split("```")[1].split("```")[0].strip()
            else:
                json_str = response.strip()

            data = json.loads(json_str)
            return data.get("relations", [])
        except Exception as e:
            logger.error(f"Error parsing concept relations: {str(e)}")
            return []


async def test_llm_client():
    """Test the LLM client"""
    client = LLMClient()

    # Test single generation
    response = await client.generate("What is a neural network?")
    print("Single response:", response[:100])

    # Test batch generation
    prompts = [
        "What is machine learning?",
        "What is deep learning?",
        "What is reinforcement learning?"
    ]
    responses = await client.generate_batch(prompts)
    print(f"\nBatch responses: {len(responses)} responses generated")

    # Test concept extraction
    texts = [
        "We propose a novel transformer architecture for graph neural networks. "
        "Our method achieves state-of-the-art results on node classification tasks."
    ]
    concepts = await client.extract_concepts_from_text(texts)
    print("\nExtracted concepts:", concepts)


if __name__ == "__main__":
    await test_llm_client()



In [ ]:
# ============================================================================
# CONFIGURATION - Adjust these limits for your use case
# ============================================================================

# Limit the number of papers to process (reduces API cost)
MAX_PAPERS = 100  # Set to None to process all papers

# Limit the total number of unique concepts/keywords extracted
MAX_CONCEPTS = 200  # Set to None for no limit

# Limit the total number of relationships (concept-concept edges)
MAX_RELATIONSHIPS = 500  # Set to None for no limit

# Keywords per paper
MAX_KEYWORDS_PER_PAPER = 8

# Output file
OUTPUT_FILE = 'arxiv_knowledge_graph.json'

print(f"Configuration:")
print(f"  Max Papers: {MAX_PAPERS or 'No limit'}")
print(f"  Max Concepts: {MAX_CONCEPTS or 'No limit'}")
print(f"  Max Relationships: {MAX_RELATIONSHIPS or 'No limit'}")
print(f"  Keywords per Paper: {MAX_KEYWORDS_PER_PAPER}")

import sys
import os

import json
import logging
import asyncio
from typing import Dict, List, Set, Tuple, Any
from collections import defaultdict
from datasets import load_dataset
import networkx as nx

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class ArXivDataLoader:
    """Load ArXiv training set from HuggingFace with provenance tracking"""

    def __init__(self):
        self.papers = []
        self.paper_metadata = {}

    def load_training_set(self, max_papers=None):
        """
        Load papers from ArXiv training set from HuggingFace
        Uses EXACT same format as the HTAG dataset

        Args:
            max_papers: Optional limit on number of papers to load

        Returns:
            Dict mapping paper_index to paper data with provenance
        """
        logger.info("Loading ArXiv training set from HuggingFace...")
        if max_papers:
            logger.info(f"Loading up to {max_papers} papers from training split")
        else:
            logger.info("Loading ALL papers from training split (no limit)")

        # Load ONLY the ArXiv CSV file (not all datasets)
        logger.info("Loading ArXiv CSV file from HTAG dataset...")
        dataset = load_dataset(
            "Cloudy1225/HTAG",
            data_files="arxiv/ArXiv.csv"
        )
        train_split = dataset["train"]

        total_papers = len(train_split)
        logger.info(f"Total training papers available: {total_papers:,}")

        papers_data = {}
        if max_papers:
            total_papers = min(total_papers, max_papers)

        logger.info(f"Processing {total_papers} papers...")

        for idx in range(total_papers):
            paper = train_split[idx]

            # Print paper keys for verification on first paper
            if idx == 0:
                logger.info("\nSample Paper Data:")
                logger.info(f"Paper: {paper}")
                logger.info(f"Keys: {list(paper.keys())}\n")

            # Use the INDEX as the paper ID to match structural data
            # The arxiv.pkl file uses indices to reference papers
            # NOT custom string IDs like "paper_{idx}"
            paper_id = idx

            # Extract text
            title = paper.get('title', '')
            abstract = paper.get('abstract', '')

            # Combine for embedding/processing
            full_text = f"{title}. {abstract}"

            # Store ONLY document content for homogeneous concept graph construction
            # Authors, citations, and FoS will be added later from structural data
            papers_data[paper_id] = {
                'id': paper_id,
                'title': title,
                'abstract': abstract,
            }

        logger.info("\n" + "="*80)
        logger.info("PAPER LENGTH STATISTICS")
        logger.info("="*80)

        logger.info(f"\nTotal papers loaded: {len(papers_data):,}")

        return papers_data


class KeywordExtractor:
    """Extract keywords from papers using LLM """

    def __init__(self, llm_client: LLMClient, max_total_concepts: int = None):
        self.llm_client = llm_client
        self.max_total_concepts = max_total_concepts

    async def extract_keywords_batch(self, papers_data: Dict[str, Dict], max_keywords: int = 8) -> Dict[str, Set[str]]:
        """
        Extract keywords from all papers using batch processing

        Returns:
            Dict mapping keyword -> set of paper IDs containing that keyword
        """
        logger.info(f"Extracting keywords from {len(papers_data)} papers...")

        keyword_to_papers = defaultdict(set)

        # ==================================================================
        # TODO: EXERCISE 1 - Implement Keyword Extraction
        # ==================================================================
        # Implement the logic to extract keywords from papers using the LLM.
        #
        # Instructions:
        # 1. Iterate through papers_data items.
        # 2. Construct a prompt for each paper that asks the LLM to extract
        #    key concepts/keywords (limit to max_keywords).
        #    Format: Ask for JSON output: {"keywords": [{"keyword": "term"}, ...]}
        # 3. Collect all prompts and use self.llm_client.generate_batch(prompts, ...) to
        #    get responses in parallel.
        # 4. Parse the JSON responses and populate the keyword_to_papers dictionary.
        #    Structure: keyword_to_papers[keyword_string].add(paper_id)
        #
        # Notes:
        # - Clean the LLM response to ensure valid JSON (handle markdown code blocks).
        # - Lowercase keywords for consistency.
        # - Respect self.max_total_concepts if set (optional but recommended).
        # ==================================================================

        print("TODO: Implement keyword extraction logic here")

        # Placeholder return for now
        return dict(keyword_to_papers)


class StructuralGraphLoader:
    """Load structural graph data (citations, authors, FoS) from ArXiv pickle"""

    @staticmethod
    def load_structural_info():
        """Download and load arxiv.pkl"""
        from huggingface_hub import hf_hub_download
        import pickle

        logger.info("Loading structural graph data from arxiv.pkl...")
        try:
            path = hf_hub_download(repo_id="Cloudy1225/HTAG", filename="arxiv/arxiv.pkl", repo_type="dataset")
            with open(path, 'rb') as f:
                data = pickle.load(f)

            logger.info("Successfully loaded structural data")
            logger.info("Structural Data Statistics:")
            for k, v in data.items():
                if hasattr(v, 'shape'):
                    logger.info(f"  {k}: shape={v.shape}")
                elif isinstance(v, tuple) and len(v) == 2 and hasattr(v[0], 'shape'):
                    logger.info(f"  {k}: tuple of shapes ({v[0].shape}, {v[1].shape})")
                else:
                    logger.info(f"  {k}: type={type(v)}")

            return data
        except Exception as e:
            logger.error(f"Failed to load structural data: {e}")
            return None



class GraphFusion:
    """
    Implementation of Graphusion algorithm for knowledge graph construction

    Based on: GraphFusion: A Multi-Source Knowledge Graph Augmentation Method
    Paper: https://arxiv.org/pdf/2407.10794

    Three-step pipeline:
    1. Seed Entity Generation: Extract keywords from documents
    2. Candidate Triplet Extraction: Extract (head, relation, tail) triplets
    3. Knowledge Graph Fusion: Merge entities, resolve conflicts, infer new triplets

    Supports 7 relation types from the paper:
    - Prerequisite_of, Used_for, Compare, Conjunction
    - Hyponym_of, Evaluate_for, Part_of
    """

    def __init__(self, llm_client: LLMClient):
        self.llm_client = llm_client
        self.model = 'gpt-4o-mini'  # Use mini for cost efficiency

        # Will be generated dynamically based on documents
        self.relations = []
        self.relation_definitions_text = ""

        # Caching for performance
        self.concept_relationship_cache = {}
        self.relation_type_cache = {}
        self.document_content_cache = {}



    async def create_knowledge_graph(
        self,
        papers_data: Dict[str, Dict],
        keywords_data: Dict[str, Set[str]],
        structural_data: Dict[str, Any] = None,
        max_relationships: int = None
    ) -> Dict[str, Any]:
        """
        Create knowledge graph using enhanced Refined_Graphusion methodology

        Args:
            papers_data: Dict mapping paper_id to paper data
            keywords_data: Dict mapping keyword to set of paper IDs
            max_relationships: Maximum number of relationships to extract (None for no limit)

        Returns:
            Knowledge graph data with provenance
        """
        logger.info(f"Creating knowledge graph with {len(keywords_data)} keywords from {len(papers_data)} papers")

        # Step 1: Load and cache document content
        self._load_document_content(papers_data, keywords_data)

        # Step 2: Generate domain-specific relation types
        await self._generate_relation_types(keywords_data)

        # Step 3: Generate candidate keyword mappings
        candidate_mappings = await self._generate_candidate_mappings(keywords_data)

        # Step 4: Extract relationships with provenance
        triplets = await self._extract_keyword_relationships_with_candidates(
            keywords_data,
            candidate_mappings
        )

        # Step 4.5: Limit relationships if specified
        if max_relationships:
            triplets = self._limit_triplets(triplets, max_relationships)
            logger.info(f"Limited to {len(triplets)} relationships")

        # Step 5: Build knowledge graph from triplets
        knowledge_graph = self._build_knowledge_graph_from_triplets(triplets)

        # Step 6: Run GraphFusion pipeline (merge, clean, resolve conflicts)
        logger.info("\n" + "="*80)
        logger.info("Running GraphFusion Pipeline (4 steps)")
        logger.info("="*80)
        knowledge_graph = await self._run_graphfusion_pipeline(knowledge_graph)
        logger.info("="*80 + "\n")

        # Step 7: Convert to output format with provenance
        graph_data = self._convert_to_output_format(knowledge_graph, keywords_data, papers_data, structural_data)

        return graph_data


    def _limit_triplets(self, triplets: List[Tuple[str, str, str, Dict[str, Any]]], max_relationships: int = None) -> List[Tuple[str, str, str, Dict[str, Any]]]:
        """Limit the number of triplets/relationships"""
        if max_relationships is None or len(triplets) <= max_relationships:
            return triplets

        logger.warning(f"Limiting {len(triplets)} triplets to {max_relationships}")

        # Keep triplets with supporting quotes (higher quality)
        triplets_with_quotes = [t for t in triplets if t[3].get('supporting_quote')]
        triplets_without_quotes = [t for t in triplets if not t[3].get('supporting_quote')]

        # Prioritize those with quotes
        if len(triplets_with_quotes) >= max_relationships:
            return triplets_with_quotes[:max_relationships]
        else:
            remaining = max_relationships - len(triplets_with_quotes)
            return triplets_with_quotes + triplets_without_quotes[:remaining]


    def _load_document_content(self, papers_data: Dict[str, Dict], keywords_data: Dict[str, Set[str]]):
        """Load and cache paper content for relationship context"""
        logger.info("Loading paper content for enhanced context...")

        for paper_id, paper_data in papers_data.items():
            # Create rich content representation
            content = {
                'label': paper_data['title'],
                'summary': paper_data['abstract'],
                'full_text': f"{paper_data['title']} {paper_data['abstract']}".strip(),
                'year': paper_data.get('year'),
                'categories': paper_data.get('categories', []),
                # Provenance
                'source': paper_data.get('source'),
                'dataset': paper_data.get('dataset'),
                'split': paper_data.get('split'),
                'index': paper_data.get('index')
            }
            self.document_content_cache[paper_id] = content

        logger.info(f"Loaded content for {len(self.document_content_cache)} papers")

    async def _generate_relation_types(self, keywords_data: Dict[str, Set[str]], num_sample_keywords: int = 50):
        """Generate domain-specific relation types based on paper content"""
        logger.info("Generating domain-specific relation types...")

        # Sample paper content
        sample_content = []
        sample_keywords = list(keywords_data.keys())[:num_sample_keywords]

        for keyword in sample_keywords:
            paper_ids = list(keywords_data[keyword])[:2]
            for paper_id in paper_ids:
                if paper_id in self.document_content_cache:
                    content = self.document_content_cache[paper_id]
                    sample_content.append(content['full_text'][:500])

        if not sample_content:
            logger.warning("No content available, using Graphusion paper relations")
            # The 7 relation types from Graphusion paper
            self.relations = [
                'Prerequisite_of',
                'Used_for',
                'Compare',
                'Conjunction',
                'Hyponym_of',  # is-a relationship
                'Evaluate_for',
                'Part_of'
            ]
            self.relation_definitions_text = "\\n".join([
                f"{chr(97+i)}) {rel}"
                for i, rel in enumerate(self.relations)
            ])
            return

        # Sample 5 papers
        import random
        random.shuffle(sample_content)
        sample_text = " | ".join(sample_content[:5])

        prompt = f"""Based on the following sample academic papers, generate 7-10 specific relationship types that would be most relevant for connecting concepts in computer science research.

Sample content: {sample_text}

Key concepts: {', '.join(sample_keywords[:20])}

Generate relationship types that are:
1. Generic and useful for academic papers
2. Actionable and meaningful
3. Suitable for knowledge graph construction

### Output Rules (CRITICAL):
- Return ONLY a valid JSON array
- Format: ["relationship1", "relationship2", ...]
- Include exactly 7-10 relationship types
- DO NOT include markdown code blocks (```json)
- Each relationship should be lowercase with hyphens (e.g., "builds-upon")

Examples: ["builds-upon", "contradicts", "validates", "extends", "implements", "relates-to", "depends-on"]"""

        try:
            response = await self.llm_client.generate(
                prompt=prompt,
                system_prompt="You are a knowledge graph expert specializing in academic research."
            )

            # Clean response
            response_text = response.strip()
            if response_text.startswith('```json'):
                response_text = response_text[7:]
            if response_text.endswith('```'):
                response_text = response_text[:-3]
            response_text = response_text.strip()

            # Parse relations
            self.relations = json.loads(response_text)

            if not isinstance(self.relations, list):
                raise ValueError(f"Relations must be a list, got: {type(self.relations)}")

            # Limit to 26 relations
            self.relations = self.relations[:26]

            # Create definitions
            self.relation_definitions_text = "\\n".join([
                f"{chr(97+i)}) {rel}"
                for i, rel in enumerate(self.relations)
            ])

            logger.info(f"Generated {len(self.relations)} relation types: {self.relations}")

        except Exception as e:
            logger.warning(f"Failed to generate relations, using Graphusion paper relations: {e}")
            # The 7 relation types from Graphusion paper
            self.relations = [
                'Prerequisite_of',
                'Used_for',
                'Compare',
                'Conjunction',
                'Hyponym_of',
                'Evaluate_for',
                'Part_of'
            ]
            self.relation_definitions_text = "\\n".join([
                f"{chr(97+i)}) {rel}"
                for i, rel in enumerate(self.relations)
            ])

    async def _generate_candidate_mappings(self, keywords_data: Dict[str, Set[str]]) -> Dict[str, List[str]]:
        """Generate candidate keyword mappings - EXACT CGPrompt implementation"""
        logger.info("Generating candidate keyword mappings...")

        concept_list = list(keywords_data.keys())
        keyword_mappings = {}

        if not concept_list or not self.relations:
            logger.warning("No concepts or relations available")
            return keyword_mappings

        # Prepare prompts for batch processing
        prompts = []
        target_concepts = []

        keyword_mapping_prompt = """### Instruction:
You are a knowledge graph expert. Given a target concept and a list of available concepts, identify which concepts could potentially be related to the target concept through any meaningful relationship.

### Target Concept:
{target_concept}

### Available Concepts:
{available_concepts}

### Available Relations:
{relations}

### Task:
From the available concepts list, select concepts that could have a meaningful relationship with the target concept using any of the available relations.

### Output Rules (CRITICAL):
- Return ONLY concept names, one per line
- DO NOT add numbering (1., 2., etc.)
- DO NOT add bullets (-, *)
- DO NOT add explanations
- Each line must contain exactly one concept name
- If no meaningful relationships exist, return: None

### Example:
concept1
concept2
concept3
"""

        logger.info(f"Preparing {len(concept_list)} candidate mapping requests...")

        for target_concept in concept_list:
            available_concepts = [c for c in concept_list if c != target_concept]

            # Limit to 200 concepts
            if len(available_concepts) > 200:
                available_concepts = available_concepts[:200]

            prompt = keyword_mapping_prompt.format(
                target_concept=target_concept,
                available_concepts=', '.join(available_concepts),
                relations=', '.join(self.relations)
            )

            prompts.append(prompt)
            target_concepts.append(target_concept)

        # Process in batches
        logger.info(f"Processing {len(prompts)} requests...")

        responses = await self.llm_client.generate_batch(
            prompts=prompts,
            system_prompt="You are a knowledge graph expert specializing in concept relationships.",
            show_progress=True
        )

        # Parse responses
        for target_concept, response_text in zip(target_concepts, responses):
            try:
                if response_text and response_text.lower() != 'none':
                    related_concepts = [c.strip() for c in response_text.split('\n')
                                      if c.strip() and c.strip() in concept_list]
                    keyword_mappings[target_concept] = related_concepts
                else:
                    keyword_mappings[target_concept] = []
            except Exception as e:
                logger.error(f"Error processing response for {target_concept}: {e}")
                keyword_mappings[target_concept] = []

        total_candidates = sum(len(candidates) for candidates in keyword_mappings.values())
        logger.info(f"Generated {total_candidates} candidate relationships")

        return keyword_mappings

    async def _extract_keyword_relationships_with_candidates(
        self,
        keywords_data: Dict[str, Set[str]],
        candidate_mappings: Dict[str, List[str]]
    ) -> List[Tuple[str, str, str, Dict[str, Any]]]:
        """Extract relationships using candidate-based approach with provenance"""
        logger.info("Extracting relationships with provenance...")

        triplets = []

        # ==================================================================
        # TODO: EXERCISE 2 - Implement Triplet Extraction
        # ==================================================================
        # Implement the logic to extract relationships (triplets) between concepts.
        #
        # Instructions:
        # 1. Iterate through candidate_mappings (maps query_concept -> list of candidates).
        # 2. For each query_concept and its candidates, retrieve the context (text from papers).
        #    Use self._get_documents_for_concept and self._get_documents_for_candidates.
        # 3. Construct a prompt that:
        #    - Provides the query concept and related concepts.
        #    - Provides the paper context.
        #    - Lists the allowed relation types (self.relations).
        #    - Asks the LLM to identify triplets (Head, Relation, Tail) and provide a supporting quote.
        # 4. Use self.llm_client.generate_batch() to process requests.
        # 5. Parse the output. Use self._parse_triplets_from_response() helper if you wish,
        #    or implement your own parsing logic.
        # 6. Store triplets in the 'triplets' list.
        #    Format: (head, relation, tail, metadata_dict)
        #    metadata_dict should contain 'source_documents', 'query_concept', 'supporting_quote'.
        # ==================================================================

        print("TODO: Implement triplet extraction logic here")

        return triplets

    def _get_documents_for_concept(
        self,
        concept: str,
        keywords_data: Dict[str, Set[str]],
        max_length: int = 2000,
        return_ids: bool = False
    ):
        """Get paper context for a concept"""
        paper_ids = keywords_data.get(concept, set())
        docs = []
        doc_ids = []

        for paper_id in list(paper_ids)[:10]:
            if paper_id in self.document_content_cache:
                docs.append(self.document_content_cache[paper_id]['full_text'])
                doc_ids.append(paper_id)

        text = self._truncate_documents(docs, max_length)
        if return_ids:
            return text, doc_ids
        return text

    def _get_documents_for_candidates(
        self,
        candidates: List[str],
        keywords_data: Dict[str, Set[str]],
        max_length: int = 2000,
        return_ids: bool = False
    ):
        """Get paper context for candidate concepts"""
        all_docs = []
        doc_ids = []

        for candidate in candidates:
            paper_ids = keywords_data.get(candidate, set())
            for paper_id in list(paper_ids)[:3]:
                if paper_id in self.document_content_cache:
                    all_docs.append(self.document_content_cache[paper_id]['full_text'])
                    doc_ids.append(paper_id)

        text = self._truncate_documents(all_docs, max_length)
        if return_ids:
            return text, doc_ids
        return text

    def _truncate_documents(self, docs: List[str], max_length: int) -> str:
        """Truncate documents to fit within token limits"""
        if not docs:
            return ""

        per_doc = max(1, max_length // len(docs))
        total_length = 0
        context = ""

        for doc in docs:
            new_length = min(max_length - total_length, len(doc))
            context += doc[:new_length] + '\\n'
            total_length += new_length
            if total_length >= max_length:
                break

        return context

    def _parse_triplets_from_response(
        self,
        response: str,
        query_concept: str,
        source_document_ids: List[str] = None
    ) -> List[Tuple[str, str, str, Dict[str, Any]]]:
        """Parse triplets from LLM response with provenance"""
        import re
        triplets = []
        lines = response.split('\n')

        for line in lines:
            line = line.strip()

            if not line or line.startswith('#'):
                continue

            # Remove prefixes using simple, robust regexes
            # Match number list: 1. or 1)
            line = re.sub(r'^\d+[.)]\s*', '', line)
            # Match bullet points: -, *, •
            line = re.sub(r'^[-*•]\s*', '', line)
            line = line.strip()

            # Look for triplet format: (concept, relation, concept) | "quote"
            if '(' in line and ')' in line:
                try:
                    quote = None
                    triplet_part = line

                    if '|' in line:
                        parts_by_pipe = line.split('|', 1)
                        triplet_part = parts_by_pipe[0].strip()
                        if len(parts_by_pipe) > 1:
                            quote_chunk = parts_by_pipe[1].strip()
                            # Extract text inside quotes
                            quote_match = re.search(r'["\'](.+?)["\']', quote_chunk)
                            if quote_match:
                                quote = quote_match.group(1)
                            else:
                                # Fallback: strip quotes if format is weird
                                quote = quote_chunk.strip(' "\'')

                    # Extract triplet from Content inside parens
                    # Note: We use \(([^)]+)\) to find text inside (...)
                    match = re.search(r'\(([^)]+)\)', triplet_part)
                    if not match:
                        continue

                    content = match.group(1)
                    parts = [part.strip() for part in content.split(',')]

                    if len(parts) != 3:
                        logger.debug(f"Skipping malformed triplet (not 3 parts): {content}")
                        continue

                    head, relation, tail = parts

                    if not head or not relation or not tail:
                        continue

                    # Create provenance metadata
                    provenance = {
                        'source_documents': source_document_ids or [],
                        'query_concept': query_concept,
                    }

                    if quote:
                        provenance['supporting_quote'] = quote

                    triplets.append((head, relation, tail, provenance))

                except Exception as e:
                    logger.warning(f"Failed to parse line '{line}': {e}")
                    continue

        return triplets

    def _build_knowledge_graph_from_triplets(
        self,
        triplets: List[Tuple[str, str, str, Dict[str, Any]]]
    ) -> nx.DiGraph:
        """Build knowledge graph from triplets with provenance"""
        knowledge_graph = nx.DiGraph()

        for head, relation, tail, provenance in triplets:
            knowledge_graph.add_edge(
                head,
                tail,
                relation=relation,
                source_documents=provenance.get('source_documents', []),
                query_concept=provenance.get('query_concept', ''),
                supporting_quote=provenance.get('supporting_quote', '')
            )

        logger.info(f"Built knowledge graph with {knowledge_graph.number_of_nodes()} nodes and {knowledge_graph.number_of_edges()} edges")
        return knowledge_graph

    async def _run_graphfusion_pipeline(self, knowledge_graph: nx.DiGraph) -> nx.DiGraph:
        """Run the complete GraphFusion pipeline (EXACT SAME as External ProviderAPI)"""
        logger.info("Starting GraphFusion pipeline...")

        # If graph is empty, return immediately
        if knowledge_graph.number_of_nodes() == 0:
            logger.warning("Empty knowledge graph, skipping GraphFusion pipeline")
            return knowledge_graph

        # Initialize merge tracking for provenance preservation
        self.concept_merge_mapping = {}  # Maps new_concept -> [old_concept1, old_concept2, ...]

        # Step 1: Merge concepts by renaming nodes
        merged_graph = await self._step1_merge_concepts(knowledge_graph)

        # Step 2: Clean duplicate edges
        cleaned_graph = self._step2_clean_duplicate_edges(merged_graph)

        # Step 3: Resolve conflicts
        final_resolutions_actions = await self._step3_resolve_conflicts(cleaned_graph)

        # Step 4: Apply resolutions
        final_graph = await self._step4_apply_resolution_conflicts(final_resolutions_actions, cleaned_graph)

        logger.info("GraphFusion pipeline completed!")
        return final_graph

    async def _step1_merge_concepts(self, graph: nx.DiGraph, batch_size: int = 100) -> nx.DiGraph:
        """Step 1: Identify and merge redundant concepts (EXACT SAME as External ProviderAPI)"""
        logger.info("Step 1: Merging key concepts...")

        # ==================================================================
        # TODO: EXERCISE 3 - Implement Concept Merging
        # ==================================================================
        # Identify and merge synonymous or redundant concepts.
        #
        # Instructions:
        # 1. Get all concepts from graph.nodes().
        # 2. Iterate through them in batches (batch_size).
        # 3. Construct a prompt for each batch asking the LLM to identify concepts
        #    that should be merged (e.g., "LSTM" and "Long Short-Term Memory").
        #    Format: JSON list of merge actions:
        #    [{"old_concepts": ["c1", "c2"], "new_concept": "c_new", "reason": "..."}, ...]
        # 4. Use self.llm_client.generate() to get the response.
        # 5. Parse the JSON and update the graph using nx.relabel_nodes().
        #    Tip: Maintain a mapping to avoid key errors if nodes are already renamed.
        #    Also update self.concept_merge_mapping for provenance.
        # ==================================================================

        print("TODO: Implement concept merging logic here")

        return graph

    def _step2_clean_duplicate_edges(self, graph: nx.DiGraph) -> nx.DiGraph:
        """Step 2: Remove duplicate edges (EXACT SAME as External ProviderAPI)"""
        logger.info("Step 2: Cleaning duplicate edges...")

        # Remove self-loops
        self_loops = list(nx.selfloop_edges(graph))
        if self_loops:
            graph.remove_edges_from(self_loops)
            logger.info(f"Removed {len(self_loops)} self-loops")

        logger.info(f"Graph after cleaning: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")
        return graph

    async def _step3_resolve_conflicts(self, graph: nx.DiGraph) -> List[str]:
        """Step 3: Resolve relation conflicts (EXACT SAME as External ProviderAPI)"""
        logger.info("Step 3: Resolving relation conflicts...")

        final_resolutions_actions = []

        # ==================================================================
        # TODO: EXERCISE 4 - Implement Conflict Resolution
        # ==================================================================
        # Identify and resolve conflicting relationships in local subgraphs.
        #
        # Instructions:
        # 1. Iterate through all nodes in the graph.
        # 2. For each node, extract its 1-hop subgraph (neighbors and edges).
        #    Use self._get_one_hop_subgraph_neighbors() helper.
        # 3. If the subgraph has connections, create a prompt asking the LLM to
        #    find conflicts (e.g., A part_of B AND A has_part B).
        #    Format: JSON list of actions: [{"action": "remove", "triplet": [...]}, ...]
        # 4. Collect prompts and use self.llm_client.generate_batch() to process.
        # 5. Return the list of resolution action strings (JSON content) to be applied in Step 4.
        # ==================================================================

        print("TODO: Implement conflict resolution logic here")

        return final_resolutions_actions

    async def _step4_apply_resolution_conflicts(self, final_resolutions_actions: List[str], graph: nx.DiGraph) -> nx.DiGraph:
        """Step 4: Apply resolution conflicts (EXACT SAME as External ProviderAPI)"""
        logger.info("Step 4: Applying resolution conflicts...")

        for resolution_actions in final_resolutions_actions:
            try:
                if resolution_actions == 'ERROR' or not resolution_actions or resolution_actions.strip() == '':
                    continue

                # Clean response
                response_text = resolution_actions.strip()
                if response_text.startswith('```json'):
                    response_text = response_text[7:]
                if response_text.endswith('```'):
                    response_text = response_text[:-3]
                response_text = response_text.strip()

                actions = json.loads(response_text)

                if not isinstance(actions, list):
                    continue

                for action in actions:
                    if not isinstance(action, dict) or 'action' not in action:
                        continue

                    if action['action'] == 'remove':
                        if 'triplet' not in action:
                            continue

                        triplet = action['triplet']
                        if len(triplet) != 3:
                            continue

                        head, relation, tail = triplet
                        if graph.has_edge(head, tail):
                            graph.remove_edge(head, tail)
                            logger.info(f"Removed edge: ({head}, {relation}, {tail})")

                    elif action['action'] == 'change':
                        if 'from' not in action or 'to' not in action:
                            continue

                        from_triplet = action['from']
                        to_triplet = action['to']

                        if len(from_triplet) != 3 or len(to_triplet) != 3:
                            continue

                        head, old_rel, tail = from_triplet
                        _, new_rel, _ = to_triplet

                        if graph.has_edge(head, tail):
                            graph[head][tail]['relation'] = new_rel
                            logger.info(f"Changed relation: ({head}, {old_rel}, {tail}) -> ({head}, {new_rel}, {tail})")

            except json.JSONDecodeError as e:
                logger.warning(f"Error parsing resolution actions JSON: {e}")
            except Exception as e:
                logger.warning(f"Error applying resolution: {e}")

        return graph

    def _get_one_hop_subgraph_neighbors(self, concept: str, graph: nx.DiGraph) -> str:
        """Get one-hop neighborhood of a concept (EXACT SAME as External ProviderAPI)"""
        if concept not in graph:
            return "No connections found"

        # Get all edges involving this concept
        outgoing = list(graph.out_edges(concept, data=True))
        incoming = list(graph.in_edges(concept, data=True))

        if not outgoing and not incoming:
            return "No direct connections"

        # Format as triplets
        triplets = []
        for src, dst, data in outgoing:
            rel = data.get('relation', 'unknown')
            triplets.append(f"({src}, {rel}, {dst})")

        for src, dst, data in incoming:
            rel = data.get('relation', 'unknown')
            triplets.append(f"({src}, {rel}, {dst})")

        return "\\n".join(triplets)

    def _convert_to_output_format(
        self,
        knowledge_graph: nx.DiGraph,
        keywords_data: Dict[str, Set[str]],
        papers_data: Dict[str, Dict],
        structural_data: Dict[str, Any] = None
    ) -> Dict[str, Any]:
        """
        Convert to output format with full provenance
        Creates a heterogeneous graph with:
        - Concept nodes (from knowledge graph)
        - Paper nodes (from papers_data)
        - Concept-concept edges (from knowledge graph)
        - Paper-concept edges (mentions, from provenance)
        """

        # 1. Extract concept nodes with provenance
        concept_nodes = []
        paper_to_concepts = defaultdict(set)  # Track which papers mention which concepts

        for node in knowledge_graph.nodes():
            # Get papers containing this concept
            # Check both current name and any original names (if merged)
            paper_ids = set()

            # First, try the current node name
            if node in keywords_data:
                paper_ids.update(keywords_data[node])

            # Then, check if this node was created by merging other concepts
            if hasattr(self, 'concept_merge_mapping') and node in self.concept_merge_mapping:
                for original_concept in self.concept_merge_mapping[node]:
                    if original_concept in keywords_data:
                        paper_ids.update(keywords_data[original_concept])

            # Track paper -> concept relationships
            for paper_id in paper_ids:
                paper_to_concepts[paper_id].add(node)

            papers_info = [papers_data[pid] for pid in paper_ids if pid in papers_data]

            concept_nodes.append({
                'id': node,
                'type': 'concept',  # Node type
                'label': node,
                'papers': list(paper_ids),
                'paper_count': len(paper_ids),
                # Provenance
                'sources': [
                    {
                        'paper_id': p['id'],
                        'title': p['title'],
                        'year': p.get('year'),
                        'dataset': p.get('dataset'),
                        'split': p.get('split')
                    }
                    for p in papers_info[:5]  # Limit to 5 papers
                ],
                # Track if this was a merged concept
                'merged_from': self.concept_merge_mapping.get(node, []) if hasattr(self, 'concept_merge_mapping') else []
            })

        # 2. Create paper nodes (identical to ArXiv dataset format)
        paper_nodes = []
        for paper_id, paper_data in papers_data.items():
            paper_nodes.append({
                'id': paper_id,
                'type': 'paper',  # Node type
                'title': paper_data['title'],
                'abstract': paper_data['abstract'],
                'year': paper_data.get('year') or (int(structural_data['paper_years'][paper_id]) if structural_data and 'paper_years' in structural_data and paper_id < len(structural_data['paper_years']) else None),
                'mag_id': paper_data.get('mag_id'),
                'label_id': paper_data.get('label_id'),
                'authors': paper_data.get('authors', []),
                'categories': paper_data.get('categories', []),
                # Provenance
                'source': paper_data.get('source'),
                'dataset': paper_data.get('dataset'),
                'split': paper_data.get('split'),
                'index': paper_data.get('index'),
                # Statistics
                'title_length': paper_data.get('title_length'),
                'abstract_length': paper_data.get('abstract_length'),
                'total_length': paper_data.get('total_length'),
                # Concepts mentioned in this paper
                'concepts_mentioned': list(paper_to_concepts.get(paper_id, set()))
            })

        # 3. Extract concept-concept edges (from knowledge graph)
        concept_edges = []
        for source, target, data in knowledge_graph.edges(data=True):
            concept_edges.append({
                'source': source,
                'target': target,
                'source_type': 'concept',
                'target_type': 'concept',
                'relation': data.get('relation', ''),
                'supporting_quote': data.get('supporting_quote', ''),
                # Provenance
                'source_documents': data.get('source_documents', []),
                'query_concept': data.get('query_concept', ''),
                'provenance': {
                    'extraction_method': 'refined_graphfusion',
                    'model': self.model,
                    'source_papers': data.get('source_documents', [])
                }
            })

        # 4. Create paper -> concept edges (mentions, from provenance)
        mention_edges = []
        for paper_id, concepts in paper_to_concepts.items():
            for concept in concepts:
                mention_edges.append({
                    'source': paper_id,
                    'target': concept,
                    'source_type': 'paper',
                    'target_type': 'concept',
                    'relation': 'mentions',
                    'provenance': {
                        'extraction_method': 'keyword_extraction',
                        'model': self.model
                    }
                })



        # 6. Add Structural Data (Authors, FoS, Citations)
        # Paper IDs are now integers matching the dataset indices
        structural_edges = []
        author_nodes = []
        fos_nodes = []

        if structural_data:
            logger.info("Adding structural nodes and edges...")

            # Paper IDs are now integers, so we can use them directly
            # papers_data keys are the actual indices from the dataset
            loaded_paper_ids = set(papers_data.keys())

            # A. Paper-Paper (Citations)
            # key: 'paper-paper', value: (src_array, dst_array)
            if 'paper-paper' in structural_data:
                src_idxs, dst_idxs = structural_data['paper-paper']

                count = 0
                for src_idx, dst_idx in zip(src_idxs, dst_idxs):
                    # Convert numpy types to Python int
                    src_idx = int(src_idx)
                    dst_idx = int(dst_idx)

                    # Only add edges between loaded papers
                    if src_idx in loaded_paper_ids and dst_idx in loaded_paper_ids:
                        structural_edges.append({
                            'source': src_idx,
                            'target': dst_idx,
                            'source_type': 'paper',
                            'target_type': 'paper',
                            'relation': 'cites',
                            'provenance': {'source': 'arxiv.pkl'}
                        })
                        count += 1
                logger.info(f"Added {count} citation edges")

            # B. Paper-Author
            # key: 'paper-author', value: (paper_idxs, author_idxs)
            # Use INTEGER IDs for authors to match HTAG dataset format
            if 'paper-author' in structural_data:
                p_idxs, a_idxs = structural_data['paper-author']

                existing_authors = set()

                count = 0
                for p_idx, a_idx in zip(p_idxs, a_idxs):
                    # Convert numpy types to Python int
                    p_idx = int(p_idx)
                    a_idx = int(a_idx)

                    if p_idx in loaded_paper_ids:
                        # Use INTEGER ID for author (matching HTAG format)
                        author_node_id = a_idx

                        # Add node if new
                        if a_idx not in existing_authors:
                            author_nodes.append({
                                'id': author_node_id,
                                'type': 'author',
                                'label': f"Author {a_idx}",
                                'index': a_idx,  # Store original index
                                'source': 'arxiv.pkl'
                            })
                            existing_authors.add(a_idx)

                        # Add edge: Author writes Paper
                        # README: "an author 'writes' a paper"
                        structural_edges.append({
                            'source': author_node_id,
                            'target': p_idx,
                            'source_type': 'author',
                            'target_type': 'paper',
                            'relation': 'writes',
                            'provenance': {'source': 'arxiv.pkl'}
                        })
                        count += 1
                logger.info(f"Added {len(author_nodes)} author nodes and {count} writes edges")

            # C. Paper-FoS
            # key: 'paper-fos', value: (paper_idxs, fos_idxs)
            # Use INTEGER IDs for FoS to match HTAG dataset format
            if 'paper-fos' in structural_data:
                p_idxs, f_idxs = structural_data['paper-fos']

                existing_fos = set()

                count = 0
                for p_idx, f_idx in zip(p_idxs, f_idxs):
                    # Convert numpy types to Python int
                    p_idx = int(p_idx)
                    f_idx = int(f_idx)

                    if p_idx in loaded_paper_ids:
                        # Use INTEGER ID for FoS (matching HTAG format)
                        fos_node_id = f_idx

                        # Add node if new
                        if f_idx not in existing_fos:
                            fos_nodes.append({
                                'id': fos_node_id,
                                'type': 'field_of_study',
                                'label': f"Field of Study {f_idx}",
                                'index': f_idx,  # Store original index
                                'source': 'arxiv.pkl'
                            })
                            existing_fos.add(f_idx)

                        # Add edge: Paper has_topic FoS
                        # README: "a paper 'has a topic of' a field of study"
                        structural_edges.append({
                            'source': p_idx,
                            'target': fos_node_id,
                            'source_type': 'paper',
                            'target_type': 'field_of_study',
                            'relation': 'has a topic of',
                            'provenance': {'source': 'arxiv.pkl'}
                        })
                        count += 1
                logger.info(f"Added {len(fos_nodes)} FoS nodes and {count} 'has a topic of' edges")

        # 7. Combine all nodes and edges
        all_nodes = concept_nodes + paper_nodes + author_nodes + fos_nodes
        all_edges = concept_edges + mention_edges + structural_edges

        logger.info(f"Created heterogeneous graph:")
        logger.info(f"  Nodes: {len(all_nodes)}")
        logger.info(f"  Edges: {len(all_edges)}")

        return {
            'nodes': all_nodes,
            'edges': all_edges,
            'metadata': {
                'num_nodes': len(all_nodes),
                'num_edges': len(all_edges),
                'num_concept_nodes': len(concept_nodes),
                'num_paper_nodes': len(paper_nodes),
                'num_concept_edges': len(concept_edges),
                'num_mention_edges': len(mention_edges),
                'relations': self.relations,
                'heterogeneous': True,
                'node_types': {
                    'concept': len(concept_nodes),
                    'paper': len(paper_nodes),
                    'author': len(author_nodes),
                    'field_of_study': len(fos_nodes)
                },
                'edge_types': {
                    'concept-concept': len(concept_edges),
                    'paper-concept': len(mention_edges),
                    'paper-paper': len([e for e in structural_edges if e['relation'] == 'cites']),
                    'author-paper': len([e for e in structural_edges if e['relation'] == 'writes']),
                    'paper-fos': len([e for e in structural_edges if e['relation'] == 'has a topic of'])
                },
                'graph_structure': {
                    'node_types': ['concept', 'paper', 'author', 'field_of_study'],
                    'edge_types': ['related_to', 'mentions', 'cites', 'writes', 'has a topic of']
                }
            }
        }

Configuration:
  Max Papers: 100
  Max Concepts: 200
  Max Relationships: 500
  Keywords per Paper: 8


NameError: name 'LLMClient' is not defined

## Exercise 3: Full Graphusion Pipeline

**Complete Pipeline:**

1. **Load Data:** Load HTAG dataset (only document content: id, title, abstract)
2. **Step 1 - Seed Generation:** Extract keywords using BERTopic/LLM
3. **Step 2 - Triplet Extraction:** Extract (head, relation, tail) triplets using Chain-of-Thought prompting
4. **Step 3 - Graph Fusion:**
   - 3.1: Entity Merging
   - 3.2: Conflict Resolution
   - 3.3: Novel Triplet Inference (optional)


In [ ]:
async def main(output='arxiv_knowledge_graph.json', limit=None, max_concepts=None, max_relationships=None):
    """Main pipeline"""

    print("="*80)
    print("Refined GraphFusion for ArXiv Dataset")
    print(f"Processing {'ALL' if limit is None else limit} papers from training set")
    print(f"Max Concepts: {max_concepts or 'No limit'}")
    print(f"Max Relationships: {max_relationships or 'No limit'}")
    print("="*80)

    # Step 1: Load ArXiv training papers
    print("\nStep 1: Loading ArXiv training set from HuggingFace...")
    loader = ArXivDataLoader()
    papers_data = loader.load_training_set(max_papers=limit)

    # Step 2: Extract keywords (8 per paper)
    print("\nStep 2: Extracting keywords from papers...")
    llm_client = LLMClient()
    extractor = KeywordExtractor(llm_client, max_total_concepts=max_concepts)
    keywords_data = await extractor.extract_keywords_batch(papers_data, max_keywords=8)

    # Step 3: Load structural data
    print("\nStep 3: Loading structural graph data...")
    structural_data = StructuralGraphLoader.load_structural_info()

    # Step 4: Create knowledge graph with provenance
    print("\nStep 4: Creating knowledge graph with provenance tracking...")
    graphfusion = GraphFusion(llm_client)
    graph_data = await graphfusion.create_knowledge_graph(
        papers_data,
        keywords_data,
        structural_data,
        max_relationships=max_relationships
    )

    # Step 5: Save results
    print(f"\nStep 5: Saving knowledge graph to {output}...")
    with open(output, 'w') as f:
        json.dump(graph_data, f, indent=2)

    print("\n" + "="*80)
    print("Heterogeneous Knowledge Graph Created Successfully!")
    print("="*80)
    print(f"\nNode Statistics:")
    print(f"  Concept nodes: {graph_data['metadata']['num_concept_nodes']}")
    print(f"  Paper nodes: {graph_data['metadata']['num_paper_nodes']}")
    if 'node_types' in graph_data:
        nt = graph_data['node_types']
        if 'author' in nt:
            print(f"  Author nodes: {nt['author']}")
        if 'field_of_study' in nt:
            print(f"  FoS nodes: {nt['field_of_study']}")
    print(f"  Total nodes: {graph_data['metadata']['num_nodes']}")

    print(f"\nEdge Statistics:")
    print(f"  Concept-concept edges: {graph_data['metadata']['num_concept_edges']}")
    print(f"  Paper-concept edges (mentions): {graph_data['metadata']['num_mention_edges']}")

    # Structural stats
    if 'edge_types' in graph_data:
        et = graph_data['edge_types']
        if 'paper-paper' in et:
            print(f"  Paper-paper edges (citations): {et['paper-paper']}")
        if 'author-paper' in et:
            print(f"  Author-paper edges (writes): {et['author-paper']}")
        if 'paper-fos' in et:
            print(f"  Paper-FoS edges (has a topic of): {et['paper-fos']}")

    print(f"  Total edges: {graph_data['metadata']['num_edges']}")

    print(f"\nGraph Structure:")
    print(f"  Node types: {', '.join(graph_data['metadata']['graph_structure']['node_types'])}")
    print(f"  Heterogeneous: {graph_data['metadata']['heterogeneous']}")

    print(f"\nRelations: {', '.join(graph_data['metadata']['relations'][:5])}...")
    print(f"\nOutput saved to: {output}")
    print("="*80)


if __name__ == "__main__":
    # Use configuration from the top of the file
    await main(
        output=OUTPUT_FILE,
        limit=MAX_PAPERS,
        max_concepts=MAX_CONCEPTS,
        max_relationships=MAX_RELATIONSHIPS
    )

## Visualization

Let's visualize the resulting heterogeneous graph.


In [ ]:
import plotly.graph_objects as go

with open('arxiv_knowledge_graph.json', 'r') as f:
    graph_data = json.load(f)

# Convert dictionary graph data back to node/edge lists for Plotly
nodes = graph_data['nodes']
edges = graph_data['edges']
import plotly.graph_objects as go

with open('arxiv_knowledge_graph.json', 'r') as f:
    graph_data = json.load(f)

# Convert dictionary graph data back to node/edge lists for Plotly
nodes = graph_data['nodes']
edges = graph_data['edges']

print(f"Visualizing {len(nodes)} nodes and {len(edges)} edges...")

# TODO: Use the plotly code from Part 1 to visualize this graph
# Hint: You'll need to map node IDs to positions using nx.spring_layout if you build a NetworkX graph first.

# can use pyvis Network instead if more convenient

print(f"Visualizing {len(nodes)} nodes and {len(edges)} edges...")

# TODO: Use the plotly code from Part 1 to visualize this graph
# Hint: You'll need to map node IDs to positions using nx.spring_layout if you build a NetworkX graph first.

# can use pyvis Network instead if more convenient


## Summary of GraphFusion Exercise

In this notebook, you have implemented key components of the Graphusion algorithm for building a knowledge graph from academic papers. Here is a recap of the tasks you've completed:

1.  **Keyword Extraction**: You implemented logic to extract key concepts and keywords from paper abstracts using an LLM. This served as the seed for our concept graph.
2.  **Triplet Extraction**: You built a pipeline to identify meaningful relationships (triplets) between these concepts, leveraging the context of the papers and a set of candidate mappings.
3.  **Concept Merging (GraphFusion Step 1)**: You addressed the issue of synonymy and redundancy by implementing a merging step that consolidates similar concepts (e.g., "AI" and "Artificial Intelligence") into single nodes.
4.  **Conflict Resolution (GraphFusion Step 3)**: You implemented a mechanism to detect and resolve logical conflicts or inconsistent relationships within the graph, ensuring the final knowledge graph is coherent and reliable.

By completing these exercises, you've gained hands-on experience with neuro-symbolic knowledge graph construction, combining the flexibility of LLMs with structured graph algorithms.

# Final Exercise: Ontology-Driven Relationship Determination

In this exercise, you will modify the KG pipeline so that a **generated ontology is used to determine relationships**,

We follow the spirit of "ontology-grounded KG construction" but **do not implement**:
- Competency questions (CQ)
- OWL/Turtle serialization
- Wikidata matching
- TBox/ABox separation

Instead, we implement a practical pipeline:

1. **Generate relation types** (already present in GraphFusion).
2. **Generate ontology** from keywords + relation types.
3. **Classify all entities** into ontology entity classes (batch).
4. **Ontology-driven relationship determination**:
   - Use entity classes + ontology domain/range constraints to shortlist candidate relations.
   - Ask the LLM to choose the best relation among those candidates (or `none`) and provide a supporting quote.
   - Validate output strictly and build triplets.

Your goal is to implement the TODOs below.

In [ ]:
import json
import logging
from typing import Dict, List, Any, Tuple, Set, Optional

import networkx as nx

logger = logging.getLogger(__name__)


class OntologyGroundedKG:
    """
    Ontology-grounded knowledge graph construction using LLM.

    NOTE: For this educational version, we skip CQ/OWL/Wikidata/TBox-ABox.
    We ONLY do: ontology generation + entity classification + ontology-driven relation choice.

    IMPORTANT: This version EXPORTS the SAME HETEROGENEOUS FORMAT as GraphFusion:
      - nodes: concept + paper (+ optional author + field_of_study)
      - edges: concept-concept + paper-concept (mentions) + structural (cites/writes/has a topic of)
      - metadata: identical keys as GraphFusion._convert_to_output_format()
    """

    def __init__(self, llm_client, model: str = "gpt-4o-mini"):
        self.llm_client = llm_client
        self.model = model
        self.ontology: Optional[Dict[str, Any]] = None

    async def create_knowledge_graph(
        self,
        papers_data: Dict[int, Dict[str, Any]],
        keywords_data: Dict[str, Set[int]],
        graph_fusion,
        structural_data: Optional[Dict[str, Any]] = None,
        max_relationships: Optional[int] = None,
        run_graphfusion_refinement: bool = True,
    ) -> Dict[str, Any]:
        """
        Simplified pipeline (ontology-driven), exported in GraphFusion heterogeneous schema:

        1) graph_fusion loads document content (should be done by caller OR we can do it here)
        2) graph_fusion generates relation types
        3) LLM generates ontology using keywords+relations (TODO A)
        4) graph_fusion generates candidate mappings
        5) ontology-driven relationship determination (TODO C)
        6) build concept graph
        7) (optional) run GraphFusion refinement
        8) export with graph_fusion._convert_to_output_format(...)  <-- SAME AS ARXIV PIPELINE
        """

        logger.info(f"Starting ontology-driven KG construction on {len(keywords_data)} keywords")

        # ------------------------------------------------------------------
        # Ensure GraphFusion has cached doc content (safe no-op if already done)
        # ------------------------------------------------------------------
        if getattr(graph_fusion, "document_content_cache", None) in (None, {}):
            logger.info("GraphFusion document cache empty; loading document content...")
            graph_fusion._load_document_content(papers_data, keywords_data)

        # ------------------------------------------------------------------
        # Stage 1: Generate relation types (existing GraphFusion behavior)
        # ------------------------------------------------------------------
        await graph_fusion._generate_relation_types(keywords_data)
        logger.info(f"Generated {len(graph_fusion.relations)} relation types")

        # ------------------------------------------------------------------
        # Stage 2: Create ontology from keywords + relations (TODO A)
        # ------------------------------------------------------------------
        self.ontology = await self._create_ontology_from_keywords_and_relations(
            keywords=list(keywords_data.keys()),
            relations=graph_fusion.relations,
        )
        logger.info(
            f"Ontology: {len(self.ontology['entity_classes'])} classes, "
            f"{len(self.ontology['relations'])} relations"
        )

        # ------------------------------------------------------------------
        # Stage 3: Candidate mappings (existing GraphFusion behavior)
        # ------------------------------------------------------------------
        candidate_mappings = await graph_fusion._generate_candidate_mappings(keywords_data)

        # ------------------------------------------------------------------
        # Stage 4: Ontology-driven relationship determination (TODO C)
        # ------------------------------------------------------------------
        triplets = await self._determine_relationships_using_ontology(
            keywords_data=keywords_data,
            candidate_mappings=candidate_mappings,
            graph_fusion=graph_fusion,
            ontology=self.ontology,
        )

        if max_relationships is not None and len(triplets) > max_relationships:
            triplets = triplets[:max_relationships]

        # ------------------------------------------------------------------
        # Stage 5: Build concept graph from triplets (GraphFusion helper)
        # ------------------------------------------------------------------
        concept_graph: nx.DiGraph = graph_fusion._build_knowledge_graph_from_triplets(triplets)

        # Optional: apply the 4-step GraphFusion refinement on the concept graph
        if run_graphfusion_refinement:
            logger.info("Running GraphFusion refinement pipeline on concept graph...")
            concept_graph = await graph_fusion._run_graphfusion_pipeline(concept_graph)

        # ------------------------------------------------------------------
        # Stage 6: Export in SAME heterogeneous schema as GraphFusion
        # ------------------------------------------------------------------
        graph_data = graph_fusion._convert_to_output_format(
            knowledge_graph=concept_graph,
            keywords_data=keywords_data,
            papers_data=papers_data,
            structural_data=structural_data,
        )

        # Attach ontology into metadata (additive; does not break existing consumers)
        graph_data.setdefault("metadata", {})
        graph_data["metadata"]["ontology"] = self.ontology
        graph_data["metadata"]["ontology_driver"] = {
            "model": self.model,
            "run_graphfusion_refinement": run_graphfusion_refinement,
        }

        return graph_data

    # ======================================================================
    # TODO A: Ontology generation robustness + validation
    # ======================================================================
    async def _create_ontology_from_keywords_and_relations(
        self,
        keywords: List[str],
        relations: List[str],
        max_attempts: int = 3,
    ) -> Dict[str, Any]:
        """
        Students implement:
        - Prompting to produce JSON ontology object
        - Robust JSON extraction (strip markdown, find braces)
        - Structure:
            * entity_classes: non-empty list[str]
            * relations: list[dict] with name/label/description/domain/range
            * relation names must be subset of provided relations
        """

        # -------------------------
        # TODO: Write your prompt
        # -------------------------
        # Requirements:
        # - Must output ONLY JSON
        # - Must include entity_classes and relations list
        # - Each relation must have: name, label, description, domain, range
        # - Relation names must come from `relations` input
        prompt = "TODO: Build ontology generation prompt here"

        last_error = None
        for attempt in range(max_attempts):
            try:
                resp = await self.llm_client.generate(
                    prompt=prompt,
                    system_prompt="You are an ontology engineer. Output only strict JSON.",
                    model=self.model,
                )

                # -------------------------
                # TODO: Clean + parse JSON
                # -------------------------
                # Hint:
                # - remove ```json fences if present
                # - find first '{' and last '}' to extract JSON object
                # - json.loads
                ontology = None  # TODO: replace

                raise NotImplementedError("TODO A not implemented")

            except Exception as e:
                last_error = e
                logger.warning(f"Ontology creation attempt {attempt+1}/{max_attempts} failed: {e}")

        raise RuntimeError(f"Failed to create ontology after {max_attempts} attempts. Last error={last_error}")

    # Optional helper students can implement
    def _validate_ontology(self, ontology: Dict[str, Any], allowed_relation_names: Set[str]) -> Dict[str, Any]:
        """
        Students implement strict validation and return normalized ontology.

        Suggested behavior:
        - Strip whitespace in class names
        - Normalize domain/range class names to match entity_classes
        - Drop relations not in allowed_relation_names OR raise error
        """
        raise NotImplementedError("TODO: implement _validate_ontology()")

    # ======================================================================
    # TODO B: Entity classification improvements (optional)
    # ======================================================================
    async def _batch_classify_entities(self, entities: List[str]) -> Dict[str, str]:
        """
        Your base code already does this.
        Students can improve by:
        - adding a 'none-of-the-above' fallback
        - adding a second-pass normalization
        - guarding against invalid outputs
        """
        raise NotImplementedError("Use your existing implementation or implement here.")

    # ======================================================================
    # TODO C (MAIN): Ontology-driven relationship determination
    # ======================================================================
    async def _determine_relationships_using_ontology(
        self,
        keywords_data: Dict[str, Set[int]],
        candidate_mappings: Dict[str, List[str]],
        graph_fusion,
        ontology: Dict[str, Any],
    ) -> List[Tuple[str, str, str, Dict[str, Any]]]:
        """
        Students implement ontology-driven relationship determination.

        Output triplet format MUST match graph_fusion._build_knowledge_graph_from_triplets:
            (head, relation, tail, {
                "source_documents": [...],
                "query_concept": "...",
                "supporting_quote": "...",
            })
        """

        # ----------------------------------------------------------
        # 1) Prepare ontology lookups
        # ----------------------------------------------------------
        relation_lookup = {}  # TODO
        entity_classes = []   # TODO

        # ----------------------------------------------------------
        # 2) Collect all entities to classify (batch)
        # ----------------------------------------------------------
        all_entities = set()  # TODO
        entity_classifications = {}  # TODO

        # ----------------------------------------------------------
        # 3) Build batch prompts for relation determination
        # ----------------------------------------------------------
        messages_batch = []
        metadata_batch = []

        for query_concept, candidates in candidate_mappings.items():
            if not candidates:
                continue

            query_docs, query_doc_ids = graph_fusion._get_documents_for_concept(
                query_concept, keywords_data, max_length=2000, return_ids=True
            )
            candidate_docs, candidate_doc_ids = graph_fusion._get_documents_for_candidates(
                candidates, keywords_data, max_length=2000, return_ids=True
            )
            source_document_ids = list(set(query_doc_ids + candidate_doc_ids))

            if not query_docs and not candidate_docs:
                continue

            query_cls = entity_classifications.get(query_concept, "Concept")

            for cand in candidates:
                cand_cls = entity_classifications.get(cand, "Concept")

                valid_relations = []  # TODO

                if not valid_relations:
                    continue

                valid_rel_list_text = "\n".join([
                    f"- {r['name']}: {r.get('description','')}"
                    for r in valid_relations
                ])

                prompt = f"""You are selecting an ontology-defined relationship between two concepts using the provided text.

Ontology relations you may choose from (ONLY these):
{valid_rel_list_text}

Concept A (query): "{query_concept}" (class: {query_cls})
Concept B (candidate): "{cand}" (class: {cand_cls})

Rules:
- Choose exactly ONE relation from the list above, or output relation="none" if no evidence.
- If relation != "none", return a short direct quote copied from the text that supports the relation.
- Output ONLY strict JSON, no markdown.

Text:
[Query concept context]
{query_docs}

[Candidate context]
{candidate_docs}

Output JSON schema:
{{
  "relation": "<chosen relation name or none>",
  "head": "<either Concept A or Concept B>",
  "tail": "<the other concept>",
  "quote": "<required if relation != none>"
}}
"""

                messages_batch.append([
                    {"role": "system", "content": "You are a careful knowledge graph relation selector. Output only strict JSON."},
                    {"role": "user", "content": prompt},
                ])
                metadata_batch.append({
                    "query_concept": query_concept,
                    "candidate": cand,
                    "source_document_ids": source_document_ids,
                    "query_cls": query_cls,
                    "cand_cls": cand_cls,
                    "valid_relation_names": [r["name"] for r in valid_relations],
                })

        # ----------------------------------------------------------
        # 4) Batch LLM call
        # ----------------------------------------------------------
        if not messages_batch:
            return []

        responses = []  # TODO

        # ----------------------------------------------------------
        # 5) Parse + validate results into triplets
        # ----------------------------------------------------------
        triplets: List[Tuple[str, str, str, Dict[str, Any]]] = []

        for resp_text, meta in zip(responses, metadata_batch):
            try:
                data = None  # TODO
                raise NotImplementedError("TODO C parsing/validation not implemented")
            except Exception as e:
                logger.warning(f"Failed to parse/validate ontology relation selection: {e}")
                continue

        return triplets

    async def _batch_llm_call(self, messages_batch: List[List[Dict[str, str]]], max_tokens: int = 500) -> List[str]:
        """Students can adapt this to their infra."""
        raise NotImplementedError("Connect to your batch inference layer here.")


# -----------------------------------------------------------------------------
# Main runner for ArXiv pipeline (ontology-driven), producing GraphFusion schema
# -----------------------------------------------------------------------------
import asyncio

async def main_ontology(
    output: str = "arxiv_knowledge_graph_ontology.json",
    limit: Optional[int] = None,
    max_concepts: Optional[int] = None,
    max_relationships: Optional[int] = None,
    keywords_per_paper: int = 8,
    include_structural: bool = True,
    run_graphfusion_refinement: bool = True,
):
    """
    Runs the ArXiv pipeline but uses OntologyGroundedKG for concept-concept relations,
    and exports the same heterogeneous JSON schema as GraphFusion.
    """

    print("=" * 80)
    print("Ontology-Driven GraphFusion for ArXiv Dataset (GraphFusion Output Schema)")
    print(f"Processing {'ALL' if limit is None else limit} papers from training set")
    print(f"Max Concepts: {max_concepts or 'No limit'}")
    print(f"Max Relationships: {max_relationships or 'No limit'}")
    print(f"Keywords per Paper: {keywords_per_paper}")
    print(f"Include Structural Graph: {include_structural}")
    print(f"Run GraphFusion Refinement: {run_graphfusion_refinement}")
    print("=" * 80)

    # Step 1: Load papers
    print("\nStep 1: Loading ArXiv training set from HuggingFace...")
    loader = ArXivDataLoader()
    papers_data = loader.load_training_set(max_papers=limit)

    # Step 2: Extract keywords (students implement EXERCISE 1)
    print("\nStep 2: Extracting keywords from papers...")
    llm_client = LLMClient()
    extractor = KeywordExtractor(llm_client, max_total_concepts=max_concepts)
    keywords_data = await extractor.extract_keywords_batch(papers_data, max_keywords=keywords_per_paper)

    # Step 3: Load structural
    print("\nStep 3: Loading structural graph data...")
    structural_data = StructuralGraphLoader.load_structural_info() if include_structural else None

    # Step 4: Prepare GraphFusion (for caching + candidate mappings + exporter)
    print("\nStep 4: Initializing GraphFusion...")
    graphfusion = GraphFusion(llm_client)
    graphfusion._load_document_content(papers_data, keywords_data)

    # Step 5: Ontology-driven KG
    print("\nStep 5: Creating ontology-driven heterogeneous knowledge graph...")
    ontology_builder = OntologyGroundedKG(llm_client, model="gpt-4o-mini")
    graph_data = await ontology_builder.create_knowledge_graph(
        papers_data=papers_data,
        keywords_data=keywords_data,
        graph_fusion=graphfusion,
        structural_data=structural_data,
        max_relationships=max_relationships,
        run_graphfusion_refinement=run_graphfusion_refinement,
    )

    # Step 6: Save
    print(f"\nStep 6: Saving knowledge graph to {output}...")
    with open(output, "w") as f:
        json.dump(graph_data, f, indent=2)

    # Print stats
    md = graph_data.get("metadata", {})
    print("\n" + "=" * 80)
    print("Ontology-Driven Heterogeneous Knowledge Graph Created Successfully!")
    print("=" * 80)
    print(f"  Total nodes: {md.get('num_nodes', len(graph_data.get('nodes', [])))}")
    print(f"  Total edges: {md.get('num_edges', len(graph_data.get('edges', [])))}")
    print(f"  Concept nodes: {md.get('num_concept_nodes', 'N/A')}")
    print(f"  Paper nodes: {md.get('num_paper_nodes', 'N/A')}")
    print(f"  Concept-concept edges: {md.get('num_concept_edges', 'N/A')}")
    print(f"  Paper-concept edges (mentions): {md.get('num_mention_edges', 'N/A')}")
    print(f"  Output saved to: {output}")
    print("=" * 80)

    return graph_data


if __name__ == "__main__":
    # Example config wiring (replace with your own constants)
    OUTPUT_FILE = "arxiv_knowledge_graph_ontology.json"
    MAX_PAPERS = 100
    MAX_CONCEPTS = 200
    MAX_RELATIONSHIPS = 500
    MAX_KEYWORDS_PER_PAPER = 8

    await main_ontology(
            output=OUTPUT_FILE,
            limit=MAX_PAPERS,
            max_concepts=MAX_CONCEPTS,
            max_relationships=MAX_RELATIONSHIPS,
            keywords_per_paper=MAX_KEYWORDS_PER_PAPER,
            include_structural=True,
            run_graphfusion_refinement=True,
    )



## Visualization

Let's visualize the resulting heterogeneous graph.


In [ ]:
import plotly.graph_objects as go

with open('arxiv_knowledge_graph.json', 'r') as f:
    graph_data = json.load(f)

# Convert dictionary graph data back to node/edge lists for Plotly
nodes = graph_data['nodes']
edges = graph_data['edges']

print(f"Visualizing {len(nodes)} nodes and {len(edges)} edges...")

# TODO: Use the plotly code from Part 1 to visualize this graph
# Hint: You'll need to map node IDs to positions using nx.spring_layout if you build a NetworkX graph first.

# can use pyvis Network instead if more convenient
